In [ ]:
import os
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm as scipy_norm

import imageio.v2 as imageio
from IPython.display import display, Video, Markdown
from io import BytesIO

from sklearn.pipeline import Pipeline
from sklearn.linear_model import Lasso, Ridge
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

import sys
sys.path.extend(["../../"])

from utils import computation_graph_linear, squared_loss_function, fit_norm2_least_square, grad_squared_loss_wrt_linear_model
from plot_utils import Arrow3D
from utils_data import (
    generate_poly_features, generate_fourier_features, generate_features,
    generate_sinusoidal_data, generate_polinomial_data, sinusoidal_fun, norm_data,
    
)

In [ ]:
display(Markdown(open("../../_macros.md").read()))

# Linear Basis Function: Overfitting, Regularization, Normalization and Cross Validation

While linear basis function models deserve their own section, we will briefly introduce them to study and introduce the concepts of overfitting, performance metrics, regularization, and cross validation which are the main point of this section.

We have seen that a linear model is refer to a model where the output can be computed from the input using a linear operator. While the presented ideas are valid for any of the four types of regression problems introduced so far, we will be focusing on the problem $f: \mathbb{R} \to \mathbb{R}$, so that we can easily plot many of these concepts. 

## Basis Functions

Linear basis function models extend a simple idea from vector spaces you have learnt in Algebra courses. In algebra lectures we are taught that a vector in some $\mathbb{R}^D$ can be written down as a linear combination of $D$ basis vectors. Where, functions can be seen as infinite dimensional vectors, and so we can write functions as linear combinations of basis functions. That's all.

A particular famous example of basis functions are the fourier series where the basis functions are sines and cosines with different frequencies and phases. Linear combinations of theses sines and cosines yield to different functions. In fact, If I do not remember bad there where some conditions that, if satisfied by a function, implied that it could be written through a fourier expansion.

In this chapter we will be working with the basis functions that correspond to polinomials of order $p$. A particular interesting thing about this is that we can work with a finite number of basis functions. Fourier expansion, for example, consider an infinite number of basis function.

Aa polinomial of order $2$, for example, can be written down by the linear combination of three basis functions:

$$
\begin{split}
\phi_0(x) = x^0\\
\phi_1(x) = x\\
\phi_2(x) = x^2\\
\end{split}
$$

In other words we have:

$$
\begin{split}
f(x) &= w_0 \cdot \phi_0(x)  + w_1 \cdot \phi_1(x)  + w_2 \cdot \phi_2(x)\\
     &= w_0 + w_1 x  + w_2 x^2
\end{split}
$$

Since they are linear combinations, they can be compactly expressed through:

$$
\begin{split}
f(x) &= \xvect\wvec \\
\xvect &= [\phi_0(x),\phi_1(x),\phi_2(x)]\\
\wvect &= [w_0,w_1,w_2]
\end{split}
$$


In [ ]:
## fix seed so that randomness is controlled.
np.random.seed(1)

## grid where basis functions and example functions are evaluated
## polynomial and fourier bases are shown over different x ranges
xmin_poly, xmax_poly = -2, 2
xmin_fourier, xmax_fourier = -5, 5
N_grid = 200
x_grid_poly = np.linspace(xmin_poly, xmax_poly, N_grid).reshape(-1, 1)
x_grid_fourier = np.linspace(xmin_fourier, xmax_fourier, N_grid).reshape(-1, 1)

poly_degree = 5
n_frequencies = 3
n_examples = 5

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

## ---------------------------------------------------------------------- ##
## Polynomial basis: the basis functions themselves, and example functions
## obtained as random linear combinations of that basis
## ---------------------------------------------------------------------- ##
Phi_poly = generate_poly_features(x_grid_poly, poly_degree, add_bias=True)

ax = axes[0, 0]
for j in range(Phi_poly.shape[1]):
    ax.plot(x_grid_poly, Phi_poly[:, j], label=fr'$\phi_{j}(x) = x^{j}$')
ax.set_title(f'Polynomial basis functions (order {poly_degree})')
ax.set_xlabel('x')
ax.legend(loc='upper right', fontsize=8)
ax.set_ylim([-4,4])

ax = axes[0, 1]
for i in range(n_examples):
    w = np.random.normal(0, 1, size=(Phi_poly.shape[1], 1))
    y = Phi_poly @ w
    ax.plot(x_grid_poly, y, label=f'sample {i+1}')
ax.set_title('Example functions: random linear combinations\nof the polynomial basis')
ax.set_xlabel('x')
ax.legend(loc='upper right', fontsize=8)
ax.set_ylim([-4,4])

## ---------------------------------------------------------------------- ##
## Fourier basis: the basis functions themselves, and example functions
## obtained as random linear combinations of that basis
## ---------------------------------------------------------------------- ##
Phi_fourier = generate_fourier_features(x_grid_fourier, n_frequencies, add_bias=True)

fourier_labels = [r'$\phi_0(x) = 1$']
for k in range(1, n_frequencies + 1):
    fourier_labels.append(fr'$\sin({k}x)$')
    fourier_labels.append(fr'$\cos({k}x)$')

ax = axes[1, 0]
for j in range(Phi_fourier.shape[1]):
    ax.plot(x_grid_fourier, Phi_fourier[:, j], label=fourier_labels[j])
ax.set_title(f'Fourier basis functions ({n_frequencies} frequencies)')
ax.set_xlabel('x')
ax.legend(loc='upper right', fontsize=8)

ax = axes[1, 1]
for i in range(n_examples):
    w = np.random.normal(0, 1, size=(Phi_fourier.shape[1], 1))
    y = Phi_fourier @ w
    ax.plot(x_grid_fourier, y, label=f'sample {i+1}')
ax.set_title('Example functions: random linear combinations\nof the fourier basis')
ax.set_xlabel('x')
ax.legend(loc='upper right', fontsize=8)

plt.tight_layout()

In [ ]:
## this cell is generated by claude based on the previous cell and the way I create the videos; with specific prompt on what to show.

## fix seed so the video reproduces the same example functions
## shown in the static plot above
np.random.seed(1)

## same grids/params as the static basis-function plot
xmin_poly, xmax_poly = -2, 2
xmin_fourier, xmax_fourier = -5, 5
N_grid = 200
x_grid_poly = np.linspace(xmin_poly, xmax_poly, N_grid).reshape(-1, 1)
x_grid_fourier = np.linspace(xmin_fourier, xmax_fourier, N_grid).reshape(-1, 1)

poly_degree = 5
n_frequencies = 3
n_examples = 5

Phi_poly = generate_poly_features(x_grid_poly, poly_degree, add_bias=True)
Phi_fourier = generate_fourier_features(x_grid_fourier, n_frequencies, add_bias=True)

## ---- 5 random full-coefficient examples ---- ##
w_poly_list = [np.random.normal(0, 1, size=(Phi_poly.shape[1], 1)) for _ in range(n_examples)]
w_fourier_list = [np.random.normal(0, 1, size=(Phi_fourier.shape[1], 1)) for _ in range(n_examples)]

poly_titles = [f'Example function {i+1}/{n_examples}' for i in range(n_examples)]
fourier_titles = [f'Example function {i+1}/{n_examples}' for i in range(n_examples)]

## ---- extra examples: some coefficients forced to zero, to show how
## restricting which basis functions are active restricts the family
## of functions we can represent (e.g. only orders 0-2 active -> parabola) ---- ##
poly_masks = [
    ([0], 'only order 0 active (constant)'),
    ([0, 1], 'only orders 0-1 active (line)'),
    ([0, 1, 2], 'only orders 0-2 active (parabola)'),
    ([0, 1, 2, 3], 'only orders 0-3 active (cubic)'),
]

fourier_masks = [
    ([0], 'only bias active (constant)'),
    ([0, 1, 2], 'only bias + freq. 1 active'),
    ([0, 1, 2, 3, 4], 'only bias + freqs. 1-2 active'),
    ([0, 1, 2, 3, 4, 5, 6], 'all frequencies active'),
]

for active_idx, desc in poly_masks:
    w = np.random.normal(0, 1, size=(Phi_poly.shape[1], 1))
    mask = np.zeros_like(w)
    mask[active_idx] = 1
    w_poly_list.append(w * mask)
    poly_titles.append(desc)

for active_idx, desc in fourier_masks:
    w = np.random.normal(0, 1, size=(Phi_fourier.shape[1], 1))
    mask = np.zeros_like(w)
    mask[active_idx] = 1
    w_fourier_list.append(w * mask)
    fourier_titles.append(desc)

## keep both columns in sync: pad the shorter list by repeating its last frame
n_total = max(len(w_poly_list), len(w_fourier_list))
while len(w_poly_list) < n_total:
    w_poly_list.append(w_poly_list[-1])
    poly_titles.append(poly_titles[-1])
while len(w_fourier_list) < n_total:
    w_fourier_list.append(w_fourier_list[-1])
    fourier_titles.append(fourier_titles[-1])

y_poly_list = [Phi_poly @ w for w in w_poly_list]
y_fourier_list = [Phi_fourier @ w for w in w_fourier_list]

## fixed axis limits so the video doesn't jitter between frames
poly_ylim = [-4, 4]
fourier_ylim = [
    min(y.min() for y in y_fourier_list) - 0.5,
    max(y.max() for y in y_fourier_list) + 0.5,
]

## ====================
## for video generation

## temporary filename
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

## ---- left column: basis functions themselves, drawn once, static ---- ##
ax = axes[0, 0]
for j in range(Phi_poly.shape[1]):
    ax.plot(x_grid_poly, Phi_poly[:, j], label=fr'$\phi_{j}(x) = x^{j}$')
ax.set_title(f'Polynomial basis functions (order {poly_degree})')
ax.set_xlabel('x')
ax.legend(loc='upper right', fontsize=8)
ax.set_ylim(poly_ylim)

fourier_labels = [r'$\phi_0(x) = 1$']
for k in range(1, n_frequencies + 1):
    fourier_labels.append(fr'$\sin({k}x)$')
    fourier_labels.append(fr'$\cos({k}x)$')

ax = axes[1, 0]
for j in range(Phi_fourier.shape[1]):
    ax.plot(x_grid_fourier, Phi_fourier[:, j], label=fourier_labels[j])
ax.set_title(f'Fourier basis functions ({n_frequencies} frequencies)')
ax.set_xlabel('x')
ax.legend(loc='upper right', fontsize=8)

## ---- right column: example functions, one at a time, animated ---- ##
ax_poly_ex = axes[0, 1]
ax_fourier_ex = axes[1, 1]

for i in range(n_total):

    w_poly_str = np.array2string(w_poly_list[i].ravel(), precision=2, separator=', ')
    w_fourier_str = np.array2string(w_fourier_list[i].ravel(), precision=2, separator=', ')

    ax_poly_ex.cla()
    ax_poly_ex.plot(x_grid_poly, y_poly_list[i], color=f'C{i % 10}')
    ax_poly_ex.set_title(f'{poly_titles[i]}\nw = {w_poly_str}', fontsize=9)
    ax_poly_ex.set_xlabel('x')
    ax_poly_ex.set_ylim(poly_ylim)

    ax_fourier_ex.cla()
    ax_fourier_ex.plot(x_grid_fourier, y_fourier_list[i], color=f'C{i % 10}')
    ax_fourier_ex.set_title(f'{fourier_titles[i]}\nw = {w_fourier_str}', fontsize=9)
    ax_fourier_ex.set_xlabel('x')
    ax_fourier_ex.set_ylim(fourier_ylim)

    fig.tight_layout()

    ## save video frame
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)

    buf.seek(0)
    frame = imageio.imread(buf)
    writer.append_data(frame)

writer.close()
plt.close()

In [ ]:
# Mostrar el video en Jupyter Notebook
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

## The lack of knowledge: Aleatoric and Epistemic Uncertainties

Let's fit some different polynomial to some data. We will generate data that follows a polynomial, and data that follows whatever other function, in this case it will be a combination of a linear and periodic functions.

To generate data that follows a polynomial of whatever order, we randomly select $x$ points from within a domain in $\mathbb{R}$, and then for each point generate:

$$
\begin{split}
t = \sum_{p=1}^P w_p x^p  + \epsilon;\,\, \epsilon \sim \mathcal{N}(\epsilon \mid 0, \sigma_\epsilon^2 )
\end{split}
$$

The other function we will use is a sinusoid with some frequency combined with a linear function. In other words:

$$
\begin{split}
t = 0.3\cdot x + \sin(fx) + \epsilon;\,\, \epsilon \sim \mathcal{N}(\epsilon \mid 0, \sigma_\epsilon^2 )
\end{split}
$$

In [ ]:
## data simulation generation
xmin = -3
xmax = 3
N_points = 30
noise_var = 0.5

# plot function
N_grid = 100
x_grid = np.linspace(xmin, xmax, N_grid)

# polinomial data
poly_degree = 2
X_train_poly, t_train_poly = generate_polinomial_data(xmin, xmax, poly_degree, noise_var, N_points, seed = 1)

# sinusoidal data
frequency = 2
X_train_sinu, t_train_sinu = generate_sinusoidal_data(xmin, xmax, frequency, noise_var, N_points, seed = 1)

# true functions
y_grid_poly = computation_graph_linear(  
                                           generate_features(x_grid, poly_degree) , 
                                           w = np.array([[2],[-1.5],[0.9]]) ,
                                           b = 0 ,
                                        )
y_grid_sinu = sinusoidal_fun(x_grid, frequency = frequency)

fig, (ax1,ax2) = plt.subplots(1,2, figsize = (10,5))


ax1.plot(X_train_poly, t_train_poly , 'x', color = 'gray', label = 'observations')
ax1.plot(x_grid, y_grid_poly, color = 'k', label = 'true function')
ax1.set_title(f'Polynomial data P = {poly_degree} \n noise variance {noise_var}')
ax1.legend()


ax2.plot(X_train_sinu, t_train_sinu , 'x', color = 'gray', label = 'observations')
ax2.plot(x_grid, y_grid_sinu, color = 'k', label = 'true function')
ax2.set_title(f'Sinusoidal data \n noise variance {noise_var}')
ax2.legend()



Let's fit some polynomial models of different orders, starting from order 1, which is a line. To fit this model, we use the squared loss function. This means that the optimal model vector $\wvec$ can be obtained directly through the ordinary least square solution:

$$
\begin{split}
\Wmat_\text{opt}=\pareinv{\Xmatt\Xmat}\Xmatt\Tmat
\end{split}
$$



In [ ]:
## Dataset specs generation
xmin = -3
xmax = 3
N_points = 30

# polinomial data
poly_degree = 2

# sinusoidal data
frequency = 2

# plot function
xmin = -3
xmax = 3
N_grid = 100
x_grid = np.linspace(xmin, xmax, N_grid)

# true functions
y_grid_poly_true = computation_graph_linear(  
                                           generate_features(x_grid, poly_degree) , 
                                           w = np.array([[2],[-1.5],[0.9]]) ,
                                           b = 0 ,
                                        )
y_grid_sinu_true = sinusoidal_fun(x_grid, frequency = frequency)

for noise_var in  [0.5]:

    # polinomial data
    X_train_poly, t_train_poly = generate_polinomial_data(xmin, xmax, poly_degree, noise_var, N_points, seed = 1)
    
    # sinusoidal data
    X_train_sinu, t_train_sinu = generate_sinusoidal_data(xmin, xmax, frequency, noise_var, N_points, seed = 1)
    
    ## plot data and true model
    fig, (ax1,ax2) = plt.subplots(1,2, figsize = (10,5))
    
    ax1.plot(X_train_poly, t_train_poly , 'x', color = 'gray', label = 'observations', markersize = 10, zorder = 10)
    ax1.plot(x_grid, y_grid_poly_true, color = 'k', label = 'true function', linewidth = 3)
    ax1.set_title(f'Polynomial data P = {poly_degree} \n noise variance {noise_var}')
    ax1.legend()
    
    
    ax2.plot(X_train_sinu, t_train_sinu , 'x', color = 'gray', label = 'observations', markersize = 10, zorder = 10)
    ax2.plot(x_grid, y_grid_sinu_true, color = 'k', label = 'true function', linewidth = 3)
    ax2.set_title(f'Sinusoidal data \n noise variance {noise_var}')
    ax2.legend()
    
    for i,poly_order in enumerate([1,2,8,20]):
        
        ## generate polynomial features
        X_feat_poly = generate_features(X_train_poly, poly_order)
        X_feat_sinu = generate_features(X_train_sinu, poly_order)
        X_feat_grid = generate_features(x_grid, poly_order)
        
        # fit model to polynomial data
        w_opt_poly = fit_norm2_least_square( X_feat_poly , t_train_poly )
        
        # fit model to sinusoidal data
        w_opt_sinu = fit_norm2_least_square( X_feat_sinu, t_train_sinu )
        
        
        # draw function on fitted model
        y_grid_poly = computation_graph_linear(  
                                               X_feat_grid, 
                                               w_opt_poly ,
                                               b = 0 ,
                                            )
        
        y_grid_sinu = computation_graph_linear(  
                                           X_feat_grid, 
                                           w_opt_sinu ,
                                           b = 0 ,
                                        )
        
        ## plotting
        ax1.plot(x_grid, y_grid_poly , linestyle = '-.' ,color = f'C{i}', label = f'order {poly_order}')
        ax1.set_ylim([-1,15])
        ax1.legend(loc = 'upper right')
    
    
        ax2.plot(x_grid, y_grid_sinu, linestyle = '-.' , color = f'C{i}', label = f'order {poly_order}')
        ax2.set_ylim([-2,1.75])
        ax2.legend(loc = 'upper right')

We observe how more expressive models fit the data better; in other words, they are able to go through each data point exactly. The question now is: what model should we choose? The more complex one that fits each training point exactly, or the less expressive one that does not go exactly through each point?

In machine learning, the optimal thing we can do is to choose a model that belongs to the family of models that has generated the data. This is what Bayes decision theory tells us (whilst there are disagreements out there when talking about the data-generating mechanism). When this does not happen, then we say the model is misspecified. 

This means that if our data is drawn from a polynomial function of order $2$ is preferable to fit a model that can recover this type of function and then model the noise as such, rather than to fit a more expressive model that fits the data perfectly. In the latter case, the model will also be fitting noise, which does belong to another part of the modelling process. We will now illustrate this concept more deeply and provide easy paths to understand it.

Machine learning is, in essence, about modelling data through modelling uncertainty or lack of knowledge about how this data was generated. There are two sources of uncertainty: aleatoric uncertainty and epistemic uncertainty.

### Aleatoric uncertainty

Aleatoric uncertainty refers to the uncertainty (noise) present in the data, which in this case is modelled through the noise distribution $\mathcal{N}(\epsilon \mid 0, \sigma_\epsilon^2 )$. Let's see what happens when we increase the noise in the data, using a lot of data, i.e when the number of training points is high.

In [ ]:
## Dataset specs generation
xmin = -3
xmax = 3
N_points = 30

# polinomial data
poly_degree = 2

# sinusoidal data
frequency = 2

# plot function
xmin = -3
xmax = 3
N_grid = 100
x_grid = np.linspace(xmin, xmax, N_grid)

# true functions
y_grid_poly_true = computation_graph_linear(  
                                           generate_features(x_grid, poly_degree) , 
                                           w = np.array([[2],[-1.5],[0.9]]) ,
                                           b = 0 ,
                                        )
y_grid_sinu_true = sinusoidal_fun(x_grid, frequency = frequency)

for noise_var in  [0.0, 0.5, 2]:

    # polinomial data
    X_train_poly, t_train_poly = generate_polinomial_data(xmin, xmax, poly_degree, noise_var, N_points, seed = 1)
    
    # sinusoidal data
    X_train_sinu, t_train_sinu = generate_sinusoidal_data(xmin, xmax, frequency, noise_var, N_points, seed = 1)
    
    ## plot data and true model
    fig, (ax1,ax2) = plt.subplots(1,2, figsize = (10,5))
    
    ax1.plot(X_train_poly, t_train_poly , 'x', color = 'gray', label = 'observations', markersize = 10, zorder = 10)
    ax1.plot(x_grid, y_grid_poly_true, color = 'k', label = 'true function', linewidth = 3)
    ax1.set_title(f'Polynomial data P = {poly_degree} \n noise variance {noise_var}')
    ax1.legend()
    
    
    ax2.plot(X_train_sinu, t_train_sinu , 'x', color = 'gray', label = 'observations', markersize = 10, zorder = 10)
    ax2.plot(x_grid, y_grid_sinu_true, color = 'k', label = 'true function', linewidth = 3)
    ax2.set_title(f'Sinusoidal data \n noise variance {noise_var}')
    ax2.legend()
    
    for i,poly_order in enumerate([1,2,8,20]):
        
        ## generate polynomial features
        X_feat_poly = generate_features(X_train_poly, poly_order)
        X_feat_sinu = generate_features(X_train_sinu, poly_order)
        X_feat_grid = generate_features(x_grid, poly_order)
        
        # fit model to polynomial data
        w_opt_poly = fit_norm2_least_square( X_feat_poly , t_train_poly )
        
        # fit model to sinusoidal data
        w_opt_sinu = fit_norm2_least_square( X_feat_sinu, t_train_sinu )
        
        
        # draw function on fitted model
        y_grid_poly = computation_graph_linear(  
                                               X_feat_grid, 
                                               w_opt_poly ,
                                               b = 0 ,
                                            )
        
        y_grid_sinu = computation_graph_linear(  
                                           X_feat_grid, 
                                           w_opt_sinu ,
                                           b = 0 ,
                                        )
        
        ## plotting
        ax1.plot(x_grid, y_grid_poly , linestyle = '-.' ,color = f'C{i}', label = f'order {poly_order}')
        ax1.set_ylim([-1,15])
        ax1.legend(loc = 'upper right')
    
    
        ax2.plot(x_grid, y_grid_sinu, linestyle = '-.' , color = f'C{i}', label = f'order {poly_order}')
        ax2.set_ylim([-2,1.75])
        ax2.legend(loc = 'upper right')

When no noise is present $\sigma_\epsilon = 0$ we observe that some models can perfectly retrieve the underlying function. We see that polynomial data can only be exactly retrieved by a polynomial of order greater than or equal to $2$. This is because a polynomial of lower order is not expressive enough to retrieve the true underlying model. For the sinusoidal data we observe something similar. Here, however, we need high-order polynomials to retrieve the true underlying function. It might be the case ( I have not checked it ) that within the family of polynomials of order $8$ or $20$ the generating function $0.3\cdot x + \sin(fx)$ is included and thus we can recover it. For this data, we are not sure, but for the polynomial data, fitting a polynomial of order equal to or greater than $2$ results in a well-specified model because the underlying function can be retrieved by the family of models we are parameterizing.

When we increase the level of noise, with $\sigma_\epsilon = 0.5$ and $\sigma_\epsilon = 2.0$ we observe how more expressive models tend to fit the data, i.e to go exactly through each point. This is the expected behaviour since the loss takes its minimum value when the model predicts the data exactly. However, we observe that the model does not retrieve a good representation of the true generative process. The exception is the polynomial of order $2$ in the polynomial data (orange lines).

In conclusion, even though the data comes from a much simpler mechanism, a more expressive model starts modelling noise as part of the functional modelling procedure. This is something not desirable, and we will see shortly how this can be measured. 

Just ask yourself, would you choose the red models to explain the data? Do you think these red models capture the underlying generative procedure that generates our data?. Note that the number of training points was big enough to correctly retrieve the underlying functions, which is observed for the examples with less noise.

### Epistemic uncertainty and model misspecification

The final observation is what happens when we have less data? Well, this is known as model uncertainty or epistemic uncertainty, which is the lack of knowledge due to data scarcity. 

To model this phenomenon, we need to generate less data. Even in the case where we do not have aleatoric uncertainty, we will observe this phenomenon. To do so, the data is generated by setting $\sigma_\epsilon=0$.

In [ ]:
## Dataset specs generation
xmin = -3
xmax = 3

# polinomial data
poly_degree = 2

# sinusoidal data
frequency = 2

# plot function
xmin = -3
xmax = 3
N_grid = 100
x_grid = np.linspace(xmin, xmax, N_grid)

# true functions
y_grid_poly_true = computation_graph_linear(  
                                           generate_features(x_grid, poly_degree) , 
                                           w = np.array([[2],[-1.5],[0.9]]) ,
                                           b = 0 ,
                                        )
y_grid_sinu_true = sinusoidal_fun(x_grid, frequency = frequency)

for N_points in  [3, 10, 100]:

    noise_var = 0.0
    
    # polinomial data
    X_train_poly, t_train_poly = generate_polinomial_data(xmin, xmax, poly_degree, noise_var, N_points, seed = 1)
    
    # sinusoidal data
    X_train_sinu, t_train_sinu = generate_sinusoidal_data(xmin, xmax, frequency, noise_var, N_points, seed = 1)
    
    ## plot data and true model
    fig, (ax1,ax2) = plt.subplots(1,2, figsize = (15,5))
    
    ax1.plot(X_train_poly, t_train_poly , 'x', color = 'k', label = 'observations', markersize = 10, zorder = 10)
    ax1.plot(x_grid, y_grid_poly_true, color = 'gray', label = 'true function', linewidth = 3)
    ax1.set_title(f'Polynomial data P = {poly_degree} \n $N$ points {N_points}')
    ax1.legend()
    
    
    ax2.plot(X_train_sinu, t_train_sinu , 'x', color = 'k', label = 'observations', markersize = 10, zorder = 10)
    ax2.plot(x_grid, y_grid_sinu_true, color = 'gray', label = 'true function', linewidth = 3)
    ax2.set_title(f'Sinusoidal data \n $N$ points {N_points}')
    ax2.legend()
    
    for i,poly_order in enumerate([1,2,8,20]):
        
        ## generate polynomial features
        X_feat_poly = generate_features(X_train_poly, poly_order)
        X_feat_sinu = generate_features(X_train_sinu, poly_order)
        X_feat_grid = generate_features(x_grid, poly_order)
        
        # fit model to polynomial data
        w_opt_poly = fit_norm2_least_square( X_feat_poly , t_train_poly )
        
        # fit model to sinusoidal data
        w_opt_sinu = fit_norm2_least_square( X_feat_sinu, t_train_sinu )
        
        
        # draw function on fitted model
        y_grid_poly = computation_graph_linear(  
                                               X_feat_grid, 
                                               w_opt_poly ,
                                               b = 0 ,
                                            )
        
        y_grid_sinu = computation_graph_linear(  
                                           X_feat_grid, 
                                           w_opt_sinu ,
                                           b = 0 ,
                                        )
        
        ## plotting
        ax1.plot(x_grid, y_grid_poly , linestyle = '-.' ,color = f'C{i}', label = f'order {poly_order}')
        ax1.set_ylim([-1,15])
        ax1.legend(loc = 'lower right')
    
    
        ax2.plot(x_grid, y_grid_sinu, linestyle = '-.' , color = f'C{i}', label = f'order {poly_order}')
        ax2.set_ylim([-2,1.75])
        ax2.legend(loc = 'lower right')

We observe a couple of interesting things. First, note there is no noise in the data, so any lack of knowledge about how the data was generated comes from not being able to retrieve the underlying function. This can be clearly seen because when more data is present, in the left column, all models explain the underlying data mechanism, the second-order polynomial, except the polynomial of order 1, which is not expressive enough to capture this functional form. The rest of the models collapse to the true underlying generative mechanism by setting high order coefficients $w$ to zero. In the case of the sinusoidal data, we need more expressive models, and so polynomials of order 1 and 2 do not fit the data. We see how all the uncertainty that comes from lack of knowledge is reduced when more knowledge is received. Thus, epistemic uncertainty is reducible.

Second, we see that in the scenario of less data, polynomials do weird things, retrieving functions that do not have to do with the data-generating process. This does not happen because the model is explaining noise, as before, because there is no noise in the data. This happens because there is not enough knowledge, and a model with more parameters needs small tuning on its parameters to fit the data exactly, which means that in unobserved regions the behaviour is unspecified. This has to do with the size of the space of hypotheses that remain consistent with the data: the family of high-order polynomials is able to fit numerous types of datasets as compared to low order polinomials and so we need more data to rule out the many other functions that are also consistent with those few points, and pin down the one that actually generated the data. From the perspective of the linear system solved through ordinary least square errors, we would have more parameters than equations and so there is an infinite number of curves that fit the data. 

Third, we see that the sinusoidal data is harder to explain by a polynomial. First of all, note that polynomial functions (although being of a high order) encode the polynomial of order two within the family of plausible functions. For the sinusoidal data, this is also true when we restrict to an interval, but high-order polynomials and more data are needed. This can be explained through the lens of Occam's Razor and marginal likelihood: the probability assigned by the polynomial family to the polynomial data is much higher than for sinusoidal data, hence collapsing with fewer points to these models. This is something similar to what happens with convolutional neural networks and images. We need less data than with a fully connected network to retrieve the underlying function that explains the data. The explanation of the previous paragraph could also be framed through the lens of Marginal Likelihood and Occams Razzor, yet is not so intuitive.

Note, however, that my arguments regarding Marginal Likelihood and Occam's Razor are not strict since we are not computing any marginal likelihood. However, the idea is simpler models when possible.

### Overfitting and underfitting

We have seen there are two types of uncertainty that need to be modelled. We see that sometimes the models might suffer from these different uncertainties when fitting the data, trying to recover the underlying functions. These problems receive and deserve their own names: overfitting and underfitting.

To continue this analysis, let's generate data from a fourth-order polynomial and fit different models. We draw data from polynomial functions of order $4$ and $8$.


In [ ]:
## Dataset specs generation
xmin = -3
xmax = 3

# polinomial data
poly_degree_1 = 4
poly_degree_2 = 8
w_true_1 = np.array([[2],[-1.5],[0.9],[-0.6],[0.5]])
w_true_2 = np.reshape(np.poly([-3, -2, -1, 0, 1, 2, 3, 4])[::-1],(poly_degree_2+1,1))

# plot function
xmin = -3
xmax = 3
N_grid = 100
x_grid = np.linspace(xmin, xmax, N_grid)

# true functions
y_grid_poly_true_1 = computation_graph_linear(  
                                           generate_features(x_grid, poly_degree_1) , 
                                           w = w_true_1 ,
                                           b = 0 ,
                                        )

y_grid_poly_true_2 = computation_graph_linear(  
                                           generate_features(x_grid, poly_degree_2) , 
                                           w = w_true_2 ,
                                           b = 0 ,
                                        )

for N_points in  [3, 10, 100]:

    noise_var = 0.0
    
    # polinomial data
    X_train_poly_1, t_train_poly_1 = generate_polinomial_data(
        xmin, 
        xmax, 
        poly_degree_1, 
        noise_var,
        N_points, 
        w_true = w_true_1, 
        seed = 1
    )
    
    # other polynomial
    X_train_poly_2, t_train_poly_2 = generate_polinomial_data(
        xmin, 
        xmax, 
        poly_degree_2, 
        noise_var,
        N_points, 
        w_true = w_true_2, 
        seed = 1
    )
     
    ## plot data and true model
    fig, (ax1,ax2) = plt.subplots(1,2, figsize = (15,5))
    
    ax1.plot(X_train_poly_1, t_train_poly_1 , 'x', color = 'k', label = 'observations', markersize = 10, zorder = 10)
    ax1.plot(x_grid, y_grid_poly_true_1, color = 'gray', label = 'true function', linewidth = 3)
    ax1.set_title(f'Polynomial data P = {poly_degree_1} \n $N$ points {N_points}')
    ax1.legend()
    
    
    ax2.plot(X_train_poly_2, t_train_poly_2 , 'x', color = 'k', label = 'observations', markersize = 10, zorder = 10)
    ax2.plot(x_grid, y_grid_poly_true_2, color = 'gray', label = 'true function', linewidth = 3)
    ax2.set_title(f'Polynomial data P = {poly_degree_2} \n $N$ points {N_points}')
    ax2.legend()
    
    for i,poly_order in enumerate([1,2,4,8,20]):
        
        ## generate polynomial features
        X_feat_poly_1 = generate_features(X_train_poly_1, poly_order)
        X_feat_poly_2 = generate_features(X_train_poly_2, poly_order)
        X_feat_grid = generate_features(x_grid, poly_order)
        
        # fit model to polynomial data
        w_opt_poly_1 = fit_norm2_least_square( X_feat_poly_1 , t_train_poly_1 )
        
        # fit model to other polynomial data
        w_opt_poly_2 = fit_norm2_least_square( X_feat_poly_2 , t_train_poly_2 )
        
        
        # draw function on fitted model
        y_grid_poly_1 = computation_graph_linear(  
                                               X_feat_grid, 
                                               w_opt_poly_1 ,
                                               b = 0 ,
                                            )
        
        y_grid_poly_2 = computation_graph_linear(  
                                               X_feat_grid, 
                                               w_opt_poly_2 ,
                                               b = 0 ,
                                            )
        
        ## plotting
        ax1.plot(x_grid, y_grid_poly_1 , linestyle = '-.' ,color = f'C{i}', label = f'order {poly_order}')
        ax1.set_ylim([-3,70])
        ax1.legend(loc = 'lower right')
    
    
        ## plotting
        ax2.plot(x_grid, y_grid_poly_2 , linestyle = '-.' ,color = f'C{i}', label = f'order {poly_order}')
        ax2.set_ylim([-150,150])
        ax2.legend(loc = 'lower right')

In general we observe that, when the order of the polynomial is smaller than the one the data has, then the curve tends to under fit the data, i.e. it does not correctly retrieve the underlying function due to lack of expresivness, and tends to somehow explan data on average. When the model is more expressive, it overfits the data, i.e, it focuses on fitting the data more than on retrieving the underlying data generative process. More data solves this problem.

This is a well-known problem in machine learning known as overfitting and underfitting. This effect is hard to diagnose from a visual point of view because data is usually high-dimensional and visualization is not possible. However, there are many things we can do to observe this effect or to select models that have the perfect tradeoff between data fitting and model complexity.

In other words, we see that some models perfectly fit the data, which obviously gives a 0 error on the training set. The question now is, do we really want this behavior?. It turns out that our target in machine learning is to capture the underlying distribution in the data so as to make predictions on unseen data. Let's mimic this behaviour and see how the function behaves on regions from the domain where it is plausible to observe new data. To do this, we just generate more data according to the same generative procedure we have used for the training data, and see which curves better explain this data.

In [ ]:
## Dataset generation
xmin = -3
xmax = 3
noise_var = 2.0
N_points = 10
N_test = 30

# polinomial data
poly_degree = 2
X_train_poly, t_train_poly = generate_polinomial_data(xmin, xmax, poly_degree, noise_var, N_points, seed = 1)
X_test_poly, t_test_poly = generate_polinomial_data(xmin, xmax, poly_degree, noise_var, N_test, seed = 5)

# sinusoidal data
frequency = 2
X_train_sinu, t_train_sinu = generate_sinusoidal_data(xmin, xmax, frequency, noise_var, N_points, seed = 1)
X_test_sinu, t_test_sinu = generate_sinusoidal_data(xmin, xmax, frequency, noise_var, N_test, seed = 5)

# plot function
xmin = -3
xmax = 3
N_grid = 100
x_grid = np.linspace(xmin, xmax, N_grid)

# true functions
y_grid_poly = computation_graph_linear(  
                                           generate_features(x_grid, poly_degree) , 
                                           w = np.array([[2],[-1.5],[0.9]]) ,
                                           b = 0 ,
                                        )
y_grid_sinu = sinusoidal_fun(x_grid, frequency = frequency)

## plot data and true model
fig, (ax1,ax2) = plt.subplots(1,2, figsize = (20,10))

ax1.plot(X_train_poly, t_train_poly , 'x', color = 'k', label = 'observations', markersize = 10, zorder = 200)
ax1.plot(X_test_poly, t_test_poly , 'o', color = 'red', label = 'prediction data', markersize = 10, zorder = 200)
ax1.plot(x_grid, y_grid_poly, color = 'k', label = 'true function', linewidth = 2)
ax1.set_title(f'Polynomial data P = {poly_degree} \n noise variance {noise_var}')
ax1.legend()


ax2.plot(X_train_sinu, t_train_sinu , 'x', color = 'k', label = 'observations', markersize = 10, zorder = 200)
ax2.plot(X_test_sinu, t_test_sinu , 'o', color = 'red', label = 'prediction data', markersize = 10, zorder = 200)
ax2.plot(x_grid, y_grid_sinu, color = 'k', label = 'true function', linewidth = 2)
ax2.set_title(f'Sinusoidal data \n noise variance {noise_var}')
ax2.legend()

for i,poly_order in enumerate([2,15]):

    ## generate polynomial features
    X_feat_poly = generate_features(X_train_poly, poly_order)
    X_feat_sinu = generate_features(X_train_sinu, poly_order)
    X_feat_grid = generate_features(x_grid, poly_order)

    # fit model to polynomial data
    w_opt_poly = fit_norm2_least_square( X_feat_poly , t_train_poly )

    # fit model to sinusoidal data
    w_opt_sinu = fit_norm2_least_square( X_feat_sinu, t_train_sinu )


    # draw function on fitted model
    y_grid_poly = computation_graph_linear(  
                                           X_feat_grid, 
                                           w_opt_poly ,
                                           b = 0 ,
                                        )

    y_grid_sinu = computation_graph_linear(  
                                       X_feat_grid, 
                                       w_opt_sinu ,
                                       b = 0 ,
                                    )

    ## plotting
    ax1.plot(x_grid, y_grid_poly ,'-.', color = f'C{i+1}', label = f'prediction function order {poly_order}')
    ax1.set_ylim([-5,20])
    ax1.legend()


    ax2.plot(x_grid, y_grid_sinu ,'-.', color = f'C{i+1}', label = f'prediction function  order {poly_order}')
    ax2.set_ylim([-10,10])
    ax2.legend()

#### Overfitting

It turns out that the more expressive model does not make good predictions on new data, yet perfectly fits the training set, as we observe in this figure. How can we measure this mathematically?. Well, we can use a performance metric. In this case, the performance metric to use is exactly the expected distance between the function and each point. In other words, the same function used for training is used to measure performance. However, we could use other functions such as the absolute distance, the mape or many more. Performance metrics are associated with Bayes decision theory and, at least by the time of writing this, will be out of the scope of this subject.

So we will quantify how well the model behaves by measuring, over both the training and test sets, the sum of squared errors. Let's plot it alongside the pictures.

In [ ]:
## Dataset generation
xmin = -3
xmax = 3
noise_var = 2.0
N_points = 10
N_test = 30

# polinomial data
poly_degree = 2
X_train_poly, t_train_poly = generate_polinomial_data(xmin, xmax, poly_degree, noise_var, N_points, seed = 1)
X_test_poly, t_test_poly = generate_polinomial_data(xmin, xmax, poly_degree, noise_var, N_test, seed = 5)

# sinusoidal data
frequency = 2
X_train_sinu, t_train_sinu = generate_sinusoidal_data(xmin, xmax, frequency, noise_var, N_points, seed = 1)
X_test_sinu, t_test_sinu = generate_sinusoidal_data(xmin, xmax, frequency, noise_var, N_test, seed = 5)

# plot function
xmin = -3
xmax = 3
N_grid = 100
x_grid = np.linspace(xmin, xmax, N_grid)

# true functions
y_grid_poly = computation_graph_linear(  
                                           generate_features(x_grid, poly_degree) , 
                                           w = np.array([[2],[-1.5],[0.9]]) ,
                                           b = 0 ,
                                        )
y_grid_sinu = sinusoidal_fun(x_grid, frequency = frequency)

## plot data and true model
fig, (ax1,ax2) = plt.subplots(1,2, figsize = (15,7.5))

ax1.plot(X_train_poly, t_train_poly , 'x', color = 'k', label = 'observations', markersize = 10, zorder = 200)
ax1.plot(X_test_poly, t_test_poly , 'o', color = 'red', label = 'prediction data', markersize = 10, zorder = 200)
ax1.plot(x_grid, y_grid_poly, color = 'k', label = 'true function', linewidth = 2)
ax1.set_title(f'Polynomial data P = {poly_degree} \n noise variance {noise_var}')
ax1.legend()


ax2.plot(X_train_sinu, t_train_sinu , 'x', color = 'k', label = 'observations', markersize = 10, zorder = 200)
ax2.plot(X_test_sinu, t_test_sinu , 'o', color = 'red', label = 'prediction data', markersize = 10, zorder = 200)
ax2.plot(x_grid, y_grid_sinu, color = 'k', label = 'true function', linewidth = 2)
ax2.set_title(f'Sinusoidal data \n noise variance {noise_var}')
ax2.legend()

for i,poly_order in enumerate([2,15]):

    ## generate polynomial features
    X_feat_train_poly = generate_features(X_train_poly, poly_order)
    X_feat_train_sinu = generate_features(X_train_sinu, poly_order)
    X_feat_test_poly = generate_features(X_test_poly, poly_order)
    X_feat_test_sinu = generate_features(X_test_sinu, poly_order)
    X_feat_grid = generate_features(x_grid, poly_order)

    # fit model to polynomial data
    w_opt_poly = fit_norm2_least_square( X_feat_train_poly , t_train_poly )

    # fit model to sinusoidal data
    w_opt_sinu = fit_norm2_least_square( X_feat_train_sinu, t_train_sinu )


    # draw function on fitted model
    y_grid_poly = computation_graph_linear(  
                                           X_feat_grid, 
                                           w_opt_poly ,
                                           b = 0 ,
                                        )

    y_grid_sinu = computation_graph_linear(  
                                       X_feat_grid, 
                                       w_opt_sinu ,
                                       b = 0 ,
                                    )
    
    ## predictions on train and test set
    y_pred_train_poly = computation_graph_linear(  
                                                   X_feat_train_poly, 
                                                   w_opt_poly ,
                                                   b = 0 ,
                                                )
    
    y_pred_train_sinu = computation_graph_linear(  
                                                   X_feat_train_sinu, 
                                                   w_opt_sinu ,
                                                   b = 0 ,
                                                )

    y_pred_test_poly = computation_graph_linear(  
                                                   X_feat_test_poly, 
                                                   w_opt_poly ,
                                                   b = 0 ,
                                                )
    
    y_pred_test_sinu = computation_graph_linear(  
                                                   X_feat_test_sinu, 
                                                   w_opt_sinu ,
                                                   b = 0 ,
                                                )
    

    ## compute loss on the train and test sets
    loss_train_poly = np.sum(squared_loss_function( y_pred_train_poly, t_train_poly ))
    loss_train_sinu = np.sum(squared_loss_function( y_pred_train_sinu, t_train_sinu ))
    
    loss_test_poly = np.sum(squared_loss_function( y_pred_test_poly, t_test_poly ))
    loss_test_sinu = np.sum(squared_loss_function( y_pred_test_sinu, t_test_sinu ))
    
    ## plotting
    ax1.plot(x_grid, y_grid_poly, '-.' , color = f'C{i+1}', label = f'prediction function order {poly_order}')
    ax1.text(x_grid[-50],-2 -2*i, f'train loss poly order {poly_order}: {loss_train_poly:.3}', color = f'C{i+1}')
    ax1.text(x_grid[-50],-2 -2*(i+1)+1, f'test loss poly order {poly_order}: {loss_test_poly:.3}', color = f'C{i+1}')
    ax1.set_ylim([-5,20])
    ax1.legend()

    ax2.plot(x_grid, y_grid_sinu, '-.' , color = f'C{i+1}', label = f'prediction function  order {poly_order}')
    ax2.text(x_grid[-50],-2 -2*i, f'train loss poly order {poly_order}: {loss_train_sinu:.3}', color = f'C{i+1}')
    ax2.text(x_grid[-50],-2 -2*(i+1)+1, f'test loss poly order {poly_order}: {loss_test_sinu:.3}', color = f'C{i+1}')
    ax2.set_ylim([-10,10])
    ax2.legend()

This effect is known as overfitting, because we fit the training data very well, without caring about how we make predictions on unseen regions. Underfitting is the opposite effect, i.e, fitting the data worse than we could, for example by using a less expressive model. We observe how the sum of squared errors over the training set is higher for the polynomial of order 2 and smaller for the polynomial of order 15. However, on the test set we observe the opposite: the less expressive model makes predictions that are closer to the unseen data.

One important fact is that recollecting more data can lead to better models, because first it is likely that we grab that from unexplored regions where a test sample might arrive, and because the model can learn the underlying pattern in the data and extrapolate correctly to other regions. In this setting, a more expressive model might not overfit. Let's first visualize this effect.

In [ ]:
## Dataset generation
xmin = -3
xmax = 3
noise_var = 2.0
N_points = 100
N_test = 30

# polinomial data
poly_degree = 2
X_train_poly, t_train_poly = generate_polinomial_data(xmin, xmax, poly_degree, noise_var, N_points, seed = 1)
X_test_poly, t_test_poly = generate_polinomial_data(xmin, xmax, poly_degree, noise_var, N_test, seed = 5)

# sinusoidal data
frequency = 2
X_train_sinu, t_train_sinu = generate_sinusoidal_data(xmin, xmax, frequency, noise_var, N_points, seed = 1)
X_test_sinu, t_test_sinu = generate_sinusoidal_data(xmin, xmax, frequency, noise_var, N_test, seed = 5)

# plot function
xmin = -3
xmax = 3
N_grid = 100
x_grid = np.linspace(xmin, xmax, N_grid)

# true functions
y_grid_poly = computation_graph_linear(  
                                           generate_features(x_grid, poly_degree) , 
                                           w = np.array([[2],[-1.5],[0.9]]) ,
                                           b = 0 ,
                                        )
y_grid_sinu = sinusoidal_fun(x_grid, frequency = frequency)

## plot data and true model
fig, (ax1,ax2) = plt.subplots(1,2, figsize = (15,7.5))

ax1.plot(X_train_poly, t_train_poly , 'x', color = 'k', label = 'observations', markersize = 10, zorder = 200)
ax1.plot(X_test_poly, t_test_poly , 'o', color = 'red', label = 'prediction data', markersize = 10, zorder = 200)
ax1.plot(x_grid, y_grid_poly, color = 'k', label = 'true function', linewidth = 2)
ax1.set_title(f'Polynomial data P = {poly_degree} \n noise variance {noise_var}')
ax1.legend()


ax2.plot(X_train_sinu, t_train_sinu , 'x', color = 'k', label = 'observations', markersize = 10, zorder = 200)
ax2.plot(X_test_sinu, t_test_sinu , 'o', color = 'red', label = 'prediction data', markersize = 10, zorder = 200)
ax2.plot(x_grid, y_grid_sinu, color = 'k', label = 'true function', linewidth = 2)
ax2.set_title(f'Sinusoidal data \n noise variance {noise_var}')
ax2.legend()

for i,poly_order in enumerate([2,15]):

    ## generate polynomial features
    X_feat_train_poly = generate_features(X_train_poly, poly_order)
    X_feat_train_sinu = generate_features(X_train_sinu, poly_order)
    X_feat_test_poly = generate_features(X_test_poly, poly_order)
    X_feat_test_sinu = generate_features(X_test_sinu, poly_order)
    X_feat_grid = generate_features(x_grid, poly_order)

    # fit model to polynomial data
    w_opt_poly = fit_norm2_least_square( X_feat_train_poly , t_train_poly )

    # fit model to sinusoidal data
    w_opt_sinu = fit_norm2_least_square( X_feat_train_sinu, t_train_sinu )


    # draw function on fitted model
    y_grid_poly = computation_graph_linear(  
                                           X_feat_grid, 
                                           w_opt_poly ,
                                           b = 0 ,
                                        )

    y_grid_sinu = computation_graph_linear(  
                                       X_feat_grid, 
                                       w_opt_sinu ,
                                       b = 0 ,
                                    )
    
    ## predictions on train and test set
    y_pred_train_poly = computation_graph_linear(  
                                                   X_feat_train_poly, 
                                                   w_opt_poly ,
                                                   b = 0 ,
                                                )
    
    y_pred_train_sinu = computation_graph_linear(  
                                                   X_feat_train_sinu, 
                                                   w_opt_sinu ,
                                                   b = 0 ,
                                                )

    y_pred_test_poly = computation_graph_linear(  
                                                   X_feat_test_poly, 
                                                   w_opt_poly ,
                                                   b = 0 ,
                                                )
    
    y_pred_test_sinu = computation_graph_linear(  
                                                   X_feat_test_sinu, 
                                                   w_opt_sinu ,
                                                   b = 0 ,
                                                )
    

    ## compute loss on the train and test sets
    loss_train_poly = np.sum(squared_loss_function( y_pred_train_poly, t_train_poly ))
    loss_train_sinu = np.sum(squared_loss_function( y_pred_train_sinu, t_train_sinu ))
    
    loss_test_poly = np.sum(squared_loss_function( y_pred_test_poly, t_test_poly ))
    loss_test_sinu = np.sum(squared_loss_function( y_pred_test_sinu, t_test_sinu ))
    
    ## plotting
    ax1.plot(x_grid, y_grid_poly, '-.' , color = f'C{i+1}', label = f'prediction function order {poly_order}')
    ax1.text(x_grid[-50],-2 -2*i, f'train loss poly order {poly_order}: {loss_train_poly:.3}', color = f'C{i+1}')
    ax1.text(x_grid[-50],-2 -2*(i+1)+1, f'test loss poly order {poly_order}: {loss_test_poly:.3}', color = f'C{i+1}')
    ax1.set_ylim([-5,20])
    ax1.legend()

    ax2.plot(x_grid, y_grid_sinu, '-.' , color = f'C{i+1}', label = f'prediction function  order {poly_order}')
    ax2.text(x_grid[-50],-2 -2*i, f'train loss poly order {poly_order}: {loss_train_sinu:.3}', color = f'C{i+1}')
    ax2.text(x_grid[-50],-2 -2*(i+1)+1, f'test loss poly order {poly_order}: {loss_test_sinu:.3}', color = f'C{i+1}')
    ax2.set_ylim([-10,10])
    ax2.legend()

We observe that now the test error of the more expressive model is reduced.

However, when extrapolating to regions where we did not observe data, these kinds of models do not work well. Bayesian approaches usually solve this problem in out-of-distribution regions. The best you can do when making predictions is say you don't know when you actually don't know.

In [ ]:
## Dataset generation
xmin = -3
xmax = 3
xmin_test = -5
xmax_test = 5
noise_var = 2.0
N_points = 100
N_test = 30

# polinomial data
poly_degree = 2
X_train_poly, t_train_poly = generate_polinomial_data(xmin, xmax, poly_degree, noise_var, N_points, seed = 1)
X_test_poly, t_test_poly = generate_polinomial_data(xmin_test, xmax_test, poly_degree, noise_var, N_test, seed = 5)

# sinusoidal data
frequency = 2
X_train_sinu, t_train_sinu = generate_sinusoidal_data(xmin, xmax, frequency, noise_var, N_points, seed = 1)
X_test_sinu, t_test_sinu = generate_sinusoidal_data(xmin_test, xmax_test, frequency, noise_var, N_test, seed = 5)

# plot function
xmin = -6
xmax = 6
N_grid = 100
x_grid = np.linspace(xmin, xmax, N_grid)

# true functions
y_grid_poly = computation_graph_linear(  
                                           generate_features(x_grid, poly_degree) , 
                                           w = np.array([[2],[-1.5],[0.9]]) ,
                                           b = 0 ,
                                        )
y_grid_sinu = sinusoidal_fun(x_grid, frequency = frequency)

## plot data and true model
fig, (ax1,ax2) = plt.subplots(1,2, figsize = (15,7.5))

ax1.plot(X_train_poly, t_train_poly , 'x', color = 'k', label = 'observations', markersize = 10, zorder = -200)
ax1.plot(X_test_poly, t_test_poly , 'o', color = 'red', label = 'prediction data', markersize = 10, zorder = -200)
ax1.plot(x_grid, y_grid_poly, color = 'k', label = 'true function', linewidth = 2)
ax1.set_title(f'Polynomial data P = {poly_degree} \n noise variance {noise_var}')
ax1.legend()


ax2.plot(X_train_sinu, t_train_sinu , 'x', color = 'k', label = 'observations', markersize = 10, zorder = -200)
ax2.plot(X_test_sinu, t_test_sinu , 'o', color = 'red', label = 'prediction data', markersize = 10, zorder = -200)
ax2.plot(x_grid, y_grid_sinu, color = 'k', label = 'true function', linewidth = 2)
ax2.set_title(f'Sinusoidal data \n noise variance {noise_var}')
ax2.legend()

for i,poly_order in enumerate([2,15]):

    ## generate polynomial features
    X_feat_train_poly = generate_features(X_train_poly, poly_order)
    X_feat_train_sinu = generate_features(X_train_sinu, poly_order)
    X_feat_test_poly = generate_features(X_test_poly, poly_order)
    X_feat_test_sinu = generate_features(X_test_sinu, poly_order)
    X_feat_grid = generate_features(x_grid, poly_order)

    # fit model to polynomial data
    w_opt_poly = fit_norm2_least_square( X_feat_train_poly , t_train_poly )

    # fit model to sinusoidal data
    w_opt_sinu = fit_norm2_least_square( X_feat_train_sinu, t_train_sinu )


    # draw function on fitted model
    y_grid_poly = computation_graph_linear(  
                                           X_feat_grid, 
                                           w_opt_poly ,
                                           b = 0 ,
                                        )

    y_grid_sinu = computation_graph_linear(  
                                       X_feat_grid, 
                                       w_opt_sinu ,
                                       b = 0 ,
                                    )
    
    ## predictions on train and test set
    y_pred_train_poly = computation_graph_linear(  
                                                   X_feat_train_poly, 
                                                   w_opt_poly ,
                                                   b = 0 ,
                                                )
    
    y_pred_train_sinu = computation_graph_linear(  
                                                   X_feat_train_sinu, 
                                                   w_opt_sinu ,
                                                   b = 0 ,
                                                )

    y_pred_test_poly = computation_graph_linear(  
                                                   X_feat_test_poly, 
                                                   w_opt_poly ,
                                                   b = 0 ,
                                                )
    
    y_pred_test_sinu = computation_graph_linear(  
                                                   X_feat_test_sinu, 
                                                   w_opt_sinu ,
                                                   b = 0 ,
                                                )
    

    ## compute loss on the train and test sets
    loss_train_poly = np.sum(squared_loss_function( y_pred_train_poly, t_train_poly ))
    loss_train_sinu = np.sum(squared_loss_function( y_pred_train_sinu, t_train_sinu ))
    
    loss_test_poly = np.sum(squared_loss_function( y_pred_test_poly, t_test_poly ))
    loss_test_sinu = np.sum(squared_loss_function( y_pred_test_sinu, t_test_sinu ))
    
    ## plotting
    ax1.plot(x_grid, y_grid_poly, ":" , color = f'C{i+1}', label = f'prediction function order {poly_order}')
    ax1.text(x_grid[-50],-2 -2*i, f'train loss poly order {poly_order}: {loss_train_poly:.3}', color = f'C{i+1}')
    ax1.text(x_grid[-50],-2 -2*(i+1)+1, f'test loss poly order {poly_order}: {loss_test_poly:.3}', color = f'C{i+1}')
    ax1.set_ylim([-5,20])
    ax1.legend()

    ax2.plot(x_grid, y_grid_sinu, ":" , color = f'C{i+1}', label = f'prediction function  order {poly_order}')
    ax2.text(x_grid[-50],-2 -2*i, f'train loss poly order {poly_order}: {loss_train_sinu:.3}', color = f'C{i+1}')
    ax2.text(x_grid[-50],-2 -2*(i+1)+1, f'test loss poly order {poly_order}: {loss_test_sinu:.3}', color = f'C{i+1}')
    ax2.set_ylim([-10,10])
    ax2.legend()

We see an error going up since we have generated test data beyond the limits where the training was created. More expressive models fail here. 

#### Underfitting

Underfitting can be easily shown by fitting a linear model to this data when compared to a polynomial of order 2.

In [ ]:
## Dataset generation
xmin = -3
xmax = 3
xmin_test = -3
xmax_test = 3
noise_var = 0.5
N_points = 100
N_test = 30

# polinomial data
poly_degree = 2
X_train_poly, t_train_poly = generate_polinomial_data(xmin, xmax, poly_degree, noise_var, N_points, seed = 1)
X_test_poly, t_test_poly = generate_polinomial_data(xmin_test, xmax_test, poly_degree, noise_var, N_test, seed = 5)

# sinusoidal data
frequency = 2
X_train_sinu, t_train_sinu = generate_sinusoidal_data(xmin, xmax, frequency, noise_var, N_points, seed = 1)
X_test_sinu, t_test_sinu = generate_sinusoidal_data(xmin_test, xmax_test, frequency, noise_var, N_test, seed = 5)

# plot function
xmin = -6
xmax = 6
N_grid = 100
x_grid = np.linspace(xmin, xmax, N_grid)

# true functions
y_grid_poly = computation_graph_linear(  
                                           generate_features(x_grid, poly_degree) , 
                                           w = np.array([[2],[-1.5],[0.9]]) ,
                                           b = 0 ,
                                        )
y_grid_sinu = sinusoidal_fun(x_grid, frequency = frequency)

## plot data and true model
fig, (ax1,ax2) = plt.subplots(1,2, figsize = (15,7.5))

ax1.plot(X_train_poly, t_train_poly , 'x', color = 'k', label = 'observations', markersize = 10, zorder = -200)
ax1.plot(X_test_poly, t_test_poly , 'o', color = 'red', label = 'prediction data', markersize = 10, zorder = -200)
ax1.plot(x_grid, y_grid_poly, color = 'k', label = 'true function', linewidth = 2)
ax1.set_title(f'Polynomial data P = {poly_degree} \n noise variance {noise_var}')
ax1.legend()


ax2.plot(X_train_sinu, t_train_sinu , 'x', color = 'k', label = 'observations', markersize = 10, zorder = -200)
ax2.plot(X_test_sinu, t_test_sinu , 'o', color = 'red', label = 'prediction data', markersize = 10, zorder = -200)
ax2.plot(x_grid, y_grid_sinu, color = 'k', label = 'true function', linewidth = 2)
ax2.set_title(f'Sinusoidal data \n noise variance {noise_var}')
ax2.legend()

for i,poly_order in enumerate([1,2,3]):

    ## generate polynomial features
    X_feat_train_poly = generate_features(X_train_poly, poly_order)
    X_feat_train_sinu = generate_features(X_train_sinu, poly_order)
    X_feat_test_poly = generate_features(X_test_poly, poly_order)
    X_feat_test_sinu = generate_features(X_test_sinu, poly_order)
    X_feat_grid = generate_features(x_grid, poly_order)

    # fit model to polynomial data
    w_opt_poly = fit_norm2_least_square( X_feat_train_poly , t_train_poly )

    # fit model to sinusoidal data
    w_opt_sinu = fit_norm2_least_square( X_feat_train_sinu, t_train_sinu )


    # draw function on fitted model
    y_grid_poly = computation_graph_linear(  
                                           X_feat_grid, 
                                           w_opt_poly ,
                                           b = 0 ,
                                        )

    y_grid_sinu = computation_graph_linear(  
                                       X_feat_grid, 
                                       w_opt_sinu ,
                                       b = 0 ,
                                    )
    
    ## predictions on train and test set
    y_pred_train_poly = computation_graph_linear(  
                                                   X_feat_train_poly, 
                                                   w_opt_poly ,
                                                   b = 0 ,
                                                )
    
    y_pred_train_sinu = computation_graph_linear(  
                                                   X_feat_train_sinu, 
                                                   w_opt_sinu ,
                                                   b = 0 ,
                                                )

    y_pred_test_poly = computation_graph_linear(  
                                                   X_feat_test_poly, 
                                                   w_opt_poly ,
                                                   b = 0 ,
                                                )
    
    y_pred_test_sinu = computation_graph_linear(  
                                                   X_feat_test_sinu, 
                                                   w_opt_sinu ,
                                                   b = 0 ,
                                                )
    

    ## compute loss on the train and test sets
    loss_train_poly = np.sum(squared_loss_function( y_pred_train_poly, t_train_poly ))
    loss_train_sinu = np.sum(squared_loss_function( y_pred_train_sinu, t_train_sinu ))
    
    loss_test_poly = np.sum(squared_loss_function( y_pred_test_poly, t_test_poly ))
    loss_test_sinu = np.sum(squared_loss_function( y_pred_test_sinu, t_test_sinu ))
    
    ## plotting
    ax1.plot(x_grid, y_grid_poly , color = f'C{i+1}', label = f'prediction function order {poly_order}')
    ax1.text(x_grid[-50],-2 -2*i, f'train loss poly order {poly_order}: {loss_train_poly:.3}', color = f'C{i+1}')
    ax1.text(x_grid[-50],-2 -2*(i+1)+1, f'test loss poly order {poly_order}: {loss_test_poly:.3}', color = f'C{i+1}')
    ax1.set_ylim([-5,20])
    ax1.legend()

    ax2.plot(x_grid, y_grid_sinu , color = f'C{i+1}', label = f'prediction function  order {poly_order}')
    ax2.text(x_grid[-50],-2 -2*i, f'train loss poly order {poly_order}: {loss_train_sinu:.3}', color = f'C{i+1}')
    ax2.text(x_grid[-50],-2 -2*(i+1)+1, f'test loss poly order {poly_order}: {loss_test_sinu:.3}', color = f'C{i+1}')
    ax2.set_ylim([-10,10])
    ax2.legend()

We observe how both training and test errors are consistently higher. 

## Regularization

The concept of regularization refers to how we can modify the learning process in a way that generalization is improved, ie, that we encourage the model not to learn the data but the underlying generating process.

There are many ways in which a model can be regularized: from the data viewpoint (data augmentation, for example), modifying the model by changing its inductive bias or reducing/augmenting its expressiveness, or by changing the underlying loss function either by setting a different observation model or placing regularizers on the parameters through a prior distribution. This is the case we will see.

The idea is to restrict the model parameters to lie within a ball of whatever radius. This ball is defined through a norm, and we will consider the $L_2$ and $L_1$ norms. We can visualize the set of points that are plausible for the model parameters to lie in depending on the norm defining the ball.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1,2, figsize = (10,5))

## Radio
r = 1  

# L1
x_L1 = [ r, 0, -r, 0,  r]
y_L1 = [ 0, r,  0, -r, 0]

# L2
theta = np.linspace(0, 2*np.pi, 400)
x_L2 = r * np.cos(theta)
y_L2 = r * np.sin(theta)

## Dibujar bolas
ax1.plot(x_L1, y_L1, color="red")       
ax1.fill(x_L1, y_L1, alpha=0.3, label = "Plausible region")

ax2.plot(x_L2, y_L2, color="blue")       
ax2.fill(x_L2, y_L2, alpha=0.3, label = 'Plausible region')          

# Ejes
ax1.axhline(0, color="black", linewidth=0.8)
ax1.axvline(0, color="black", linewidth=0.8)
ax2.axhline(0, color="black", linewidth=0.8)
ax2.axvline(0, color="black", linewidth=0.8)

# Titulos
ax1.set_title(f"$L_1$ ball with radius {r}")
ax2.set_title(f"$L_2$ ball with radius {r}")

# Figura con aspecto cuadrado
ax1.set_xlim(-r-0.5, r+0.5)
ax1.set_ylim(-r-0.5, r+0.5)
ax2.set_xlim(-r-0.5, r+0.5)
ax2.set_ylim(-r-0.5, r+0.5)

ax1.legend()
ax2.legend()

### $L_2$ regularization

Restricting this parameter to lie within this ball can be done through a Lagrange multiplier. Note that any loss function, such as the quadratic loss, can be modified in such a way to satisfy this restriction. In particular, the modified loss can be written down as:

$$
\begin{split}
L(\Xmat,\Tmat,\Wmat) = ||(\Tmat-\Xmat\Wmat)||^2_2; \text{  subject to  } ||\Wmat||_2 \leq r
\end{split}
$$

This is equivalent to minimizing the following loss:

$$
\begin{split}
L(\Xmat,\Tmat,\Wmat) = ||(\Tmat-\Xmat\Wmat)||^2_2 + \lambda ||\Wmat||^2_2
\end{split}
$$


Up to this moment, I haven't worked out how to reach this loss function from the above restriction. However, it can be easily seen how this loss imposes the restriction. First, note that instead of using the norm, we are using the square of the norm. Since the norm is positive, the square of the norm is a monotonic function, and thus it will not change the order of points with decreasing/incresing norm. Optimizing the square of the norm has other advantages, which include:

* An analytical solution to the optimization problem.
* Is easier to optimize since the gradient of the norm is not defined at $w=0$.

So adding this term to a minimization process will force the norm of the model to be small. The hyperparameter $\lambda$ controls how restrictive we are. Big values for $\lambda$ will encourage the norm to be big, and thus the parameters will be forced to be small in order to minimize the norm. Note that this regularization, by itself, is minimized when all the weights are $0$. However, the other part of the loss function encourages the parameters to fit the data, and so minimizing this loss function is a trade-off between explaining the data and making the weights small.

The next question is why this avoids overfitting?. Well, if we force the parameters of the models to be small, then none of the dimensions of the problem might dominate over the others, or at least there is no freedom to make a parameter arbitrarily big wrt the others. This makes that high order polynomials, for example, will have it "hard" to generate wiggle functions. Note that, as we saw, the high-order coefficients are in charge of learning that wiggle behaviour that made the function go exactly through each point. If we restrict the parameters of this contribution to be small, then the model does not have that freedom to learn highly wiggling functions that explain the data.

With $L_2$ regularization, we can obtain an optimal solution for the parameters. Since the regularization term is added to the data fit term, we just need to work out the differential on this term and then sum it with the other. In this case, no chain rule is necessary. Again, writing the Frobenius norm through traces, we have:

$$
\begin{split}
\dd \lambda ||\Wmat||^2_2 &= \dd \lambda\tr{\Wmatt\Wmat}\\
&= \lambda \tr{ \dd \Wmatt\Wmat + \Wmatt \dd \Wmat} \\
&= \lambda \tr{ \Wmatt\dd\Wmat + \Wmatt \dd \Wmat} \\
&= 2\lambda \tr{\Wmatt \dd \Wmat} \\
&= 2\lambda \braT{\vvec \Wmat} \dd\vvec\Wmat
\end{split}
$$

from where the Jacobian is given by $2\lambda \braT{\vvec \Wmat}$. The corresponding gradient is just the transpose, and so $2\lambda [\vvec\Wmat]$.

We can now add this term to our previous gradient and set it to zero. 

$$
\begin{split}
-2 [\Imat \otimes \Xmatt]\vvec \bra{\Tmat - \Xmat\Wmat} + 2\lambda [\vvec\Wmat]&= 0\\
-[\Imat \otimes \Xmatt]\vvec[\Tmat] + [\Imat \otimes \Xmatt]\vvec[\Xmat\Wmat\Imat]+ \lambda [\vvec \Wmat]&= 0\\
-[\Imat \otimes \Xmatt]\vvec[\Tmat] + [\Imat \otimes \Xmatt](\Imat\otimes\Xmat)\vvec \Wmat + \lambda [\vvec \Wmat] &= 0\\
-[\Imat \otimes \Xmatt]\vvec[\Tmat] + [\Imat \otimes \Xmatt\Xmat]\vvec[\Wmat]+ \lambda [\vvec \Wmat] &= 0\\
[\Imat \otimes \Xmatt]\vvec[\Tmat]  &= [\Imat \otimes \Xmatt\Xmat]\vvec \Wmat + \lambda [\vvec \Wmat]\\
\pareinv{[\Imat \otimes \Xmatt\Xmat]+ \lambda \Imat}[\Imat \otimes \Xmatt]\vvec \Tmat  &= \vvec \Wmat\\
[\Imat \otimes \pareinv{\Xmatt\Xmat + \lambda \Imat}][\Imat \otimes \Xmatt]\vvec[\Tmat]  &= \vvec[\Wmat]\\
[\Imat \otimes \pareinv{\Xmatt\Xmat+ \lambda \Imat}\Xmatt]\vvec[\Tmat]  &= \vvec[\Wmat]\\
\vvec[\pareinv{\Xmatt\Xmat+ \lambda \Imat}\Xmatt\Tmat\Imat]  &= \vvec[\Wmat]\\
\vvec[\pareinv{\Xmatt\Xmat+ \lambda \Imat}\Xmatt\Tmat]  &= \vvec[\Wmat]
\end{split}
$$

from where:

$$
\begin{split}
\Wmat_\text{opt} = \pareinv{\Xmatt\Xmat+ \lambda \Imat}\Xmatt\Tmat
\end{split}
$$

Obviously, for one-dimensional multivariate regression $f:\mathbb{R}^N \to \mathbb{R}$, the derivation is simpler because the vec operator and Kronecker products do not take part in the derivation. This is left as an exercise.

Let's visualize the effect of the regularizer.

In [ ]:
%matplotlib inline
plt.close("all")

## Dataset generation
xmin = -3
xmax = 3
noise_var = 1
N_points = 30

# polinomial data
poly_degree = 2
X_train_poly, t_train_poly = generate_polinomial_data(xmin, xmax, poly_degree, noise_var, N_points, seed = 1)

# sinusoidal data
frequency = 2
X_train_sinu, t_train_sinu = generate_sinusoidal_data(xmin, xmax, frequency, noise_var, N_points, seed = 1)

# plot function
xmin = -3
xmax = 3
N_grid = 100
x_grid = np.linspace(xmin, xmax, N_grid)

# true functions
y_grid_poly_true = computation_graph_linear(  
                                           generate_features(x_grid, poly_degree) , 
                                           w = np.array([[2],[-1.5],[0.9]]) ,
                                           b = 0 ,
                                        )
y_grid_sinu_true = sinusoidal_fun(x_grid, frequency = frequency)

for lam in [0.0,10]:

    ## plot data and true model
    fig, (ax1,ax2) = plt.subplots(1,2, figsize = (10,5))

    ax1.plot(X_train_poly, t_train_poly , 'x', color = 'k', label = 'observations', markersize = 10)
    ax1.plot(x_grid, y_grid_poly_true, color = 'gray', label = 'true function')
    ax1.set_title(f'Polynomial data P = {poly_degree} \n $\lambda$ {lam}')
    ax1.legend()


    ax2.plot(X_train_sinu, t_train_sinu , 'x', color = 'k', label = 'observations', markersize = 10)
    ax2.plot(x_grid, y_grid_sinu_true, color = 'gray', label = 'true function')
    ax2.set_title(f'Sinusoidal data \n $\lambda$ {lam}')
    ax2.legend()

    for i,poly_order in enumerate([2,8,20]):

        ## generate polynomial features
        X_feat_poly = generate_features(X_train_poly, poly_order)
        X_feat_sinu = generate_features(X_train_sinu, poly_order)
        X_feat_grid = generate_features(x_grid, poly_order)

        # fit model to polynomial data
        w_opt_poly = fit_norm2_least_square( X_feat_poly , t_train_poly, lam = lam )

        # fit model to sinusoidal data
        w_opt_sinu = fit_norm2_least_square( X_feat_sinu, t_train_sinu, lam = lam )


        # draw function on fitted model
        y_grid_poly = computation_graph_linear(  
                                               X_feat_grid, 
                                               w_opt_poly ,
                                               b = 0 ,
                                            )

        y_grid_sinu = computation_graph_linear(  
                                           X_feat_grid, 
                                           w_opt_sinu ,
                                           b = 0 ,
                                        )

        ## plotting
        ax1.plot(x_grid, y_grid_poly,'-.' , color = f'C{i+1}', label = f'order {poly_order}')
        ax1.set_ylim([-1,16])
        ax1.legend()

        ax2.plot(x_grid, y_grid_sinu,'-.' , color = f'C{i+1}', label = f'order {poly_order}')
        ax2.set_ylim([-2,2])
        ax2.legend()
        
        print(f"Lambda {lam} poly order {poly_order} weights {[f'{float(w.item()):.4f}' for w in w_opt_poly]}")
    print("----------------------------")

We see how, by encouraging the weights to have a small norm, high-level features might be relaxed, and overfitting is reduced. This is a consequence of the parameters of the high-order polynomials having bigger values, as we saw in the above runs. Thus, regularization forces these values to be reduced. Obviously, grabbing more data reduces the effect further, since we reduce the epistemic uncertainty.

In [ ]:
%matplotlib inline
plt.close("all")

## Dataset generation
xmin = -3
xmax = 3
noise_var = 1
N_points = 100

# polinomial data
poly_degree = 2
X_train_poly, t_train_poly = generate_polinomial_data(xmin, xmax, poly_degree, noise_var, N_points, seed = 1)

# sinusoidal data
frequency = 2
X_train_sinu, t_train_sinu = generate_sinusoidal_data(xmin, xmax, frequency, noise_var, N_points, seed = 1)

# plot function
xmin = -3
xmax = 3
N_grid = 100
x_grid = np.linspace(xmin, xmax, N_grid)

# true functions
y_grid_poly_true = computation_graph_linear(  
                                           generate_features(x_grid, poly_degree) , 
                                           w = np.array([[2],[-1.5],[0.9]]) ,
                                           b = 0 ,
                                        )
y_grid_sinu_true = sinusoidal_fun(x_grid, frequency = frequency)

for lam in [0.0,10]:

    ## plot data and true model
    fig, (ax1,ax2) = plt.subplots(1,2, figsize = (10,5))

    ax1.plot(X_train_poly, t_train_poly , 'x', color = 'k', label = 'observations', markersize = 10)
    ax1.plot(x_grid, y_grid_poly_true, color = 'gray', label = 'true function')
    ax1.set_title(f'Polynomial data P = {poly_degree} \n $\lambda$ {lam}')
    ax1.legend()


    ax2.plot(X_train_sinu, t_train_sinu , 'x', color = 'k', label = 'observations', markersize = 10)
    ax2.plot(x_grid, y_grid_sinu_true, color = 'gray', label = 'true function')
    ax2.set_title(f'Sinusoidal data \n $\lambda$ {lam}')
    ax2.legend()

    for i,poly_order in enumerate([2,8,20]):

        ## generate polynomial features
        X_feat_poly = generate_features(X_train_poly, poly_order)
        X_feat_sinu = generate_features(X_train_sinu, poly_order)
        X_feat_grid = generate_features(x_grid, poly_order)

        # fit model to polynomial data
        w_opt_poly = fit_norm2_least_square( X_feat_poly , t_train_poly, lam = lam )

        # fit model to sinusoidal data
        w_opt_sinu = fit_norm2_least_square( X_feat_sinu, t_train_sinu, lam = lam )


        # draw function on fitted model
        y_grid_poly = computation_graph_linear(  
                                               X_feat_grid, 
                                               w_opt_poly ,
                                               b = 0 ,
                                            )

        y_grid_sinu = computation_graph_linear(  
                                           X_feat_grid, 
                                           w_opt_sinu ,
                                           b = 0 ,
                                        )

        ## plotting
        ax1.plot(x_grid, y_grid_poly,'-.' , color = f'C{i+1}', label = f'order {poly_order}')
        ax1.set_ylim([-1,16])
        ax1.legend()

        ax2.plot(x_grid, y_grid_sinu,'-.' , color = f'C{i+1}', label = f'order {poly_order}')
        ax2.set_ylim([-2,2])
        ax2.legend()
        
        print(f"Lambda {lam} poly order {poly_order} weights {[f'{float(w):.4f}' for w in w_opt_poly]}")
    print("----------------------------")

### Feature normalization


While normalization deserves its own chapter in the context of parameter regularization and optimization convergence, it can help understand why it is also important for good convergence and learning of the model. Normalization is the process of getting some features and putting them within the same range. Overall, it helps the learning process and also makes the model less prone to overfitting. Polynomial regression is an easy example of why.

Consider the linear combination implemented by a linear model:

$$
\begin{split}
    y = w_0 + w_1x_1 + w_2x_2 + w_3x_3 + w_4x_4
\end{split}
$$

what would happen if the scale of $x_4$ is much bigger than that of $x_1$?. This would imply that the linear combination is dominated by the $x_4$ feature. If the relative importance is not so high, learning to adapt the rest of the weights will take time. Moreover, the gradient scale will have a different impact on the learning process, and so vanilla gradient descent will fail to successfully train "fast". Also, the optimization procedure is more sensitive to the initialization of the parameters within the gradient descent algorithm. For instance, if we initialize all the parameters around $0$, then the optimization will take longer to successfully raise the parameters $w_1,w_2,w_3$ so that the relative importance of these features wrt $x_4$ is correctly calibrated. Moreover, if for whatever reason a high value of this feature encourages the model to learn some part of the data better, then the model can get stuck on making this contribution bigger, ending up in overfitting. This is, in fact, what happens in polynomial regression, since here we have:

$$
\begin{split}
    y = w_0 + w_1x + w_2x^2 + w_3x^3 + w_4x^4
\end{split}
$$

Since $x^4$ is much bigger than $x$, its contribution is bigger. This implies that at the beginning of the learning process, the model is prone to model high-order features, which are wavy and can learn to fit the data easily. Even a regularized model that makes the weights go to zero will get stuck. Why? Because even a small value for $w_4$ can be manifested as a huge contribution to $y$ due to $x^4$ being bigger than, for example, $x$; so even if the regularizer makes $w_4$ go to $0$ the effect will not disappear, because this reduction in magnitude will be compensated by $x^4$. At the same time, the rest of the coefficients will be made small by the regularizer, and the overall contribution will be given by  $x^4$. To avoid this, we can just normalize the data by subtracting the mean and dividing by the standard deviation. Let's compare the behaviour of regularized and non-regularized models, comparing data and non-data normalization.

In machine learning, there are two options. For evaluation, we can just normalize data and perform all computations in normalized space. For predictions, we usually normalize data, perform prediction, and unnormalize the output. Since normalization is a linear operator, it will not change the shape of the function learnt. 

Moreover, it can be shown that without regularization, feature normalization is not relevant when optimizing a linear model through exact minimization via least squares with an $L_2$ norm. **Exercise**

Let's visualize the effect of training over normalized features. In this case, we will be normalizing both targets $\tvec$ and features $\xvec$. We observe how normalization improves the polynomial data fits (left figure).

In [ ]:
%matplotlib inline
plt.close("all")

## Dataset generation
xmin = -3
xmax = 3
noise_var = 1
N_points = 30

# polinomial data
poly_degree = 2
_X_train_poly, _t_train_poly = generate_polinomial_data(xmin, xmax, poly_degree, noise_var, N_points, seed = 1)

# sinusoidal data
frequency = 2
_X_train_sinu, _t_train_sinu = generate_sinusoidal_data(xmin, xmax, frequency, noise_var, N_points, seed = 1)

# plot function
xmin = -3
xmax = 3
N_grid = 100
x_grid = np.linspace(xmin, xmax, N_grid)

# true functions
y_grid_poly_true = computation_graph_linear(  
                                           generate_features(x_grid, poly_degree) , 
                                           w = np.array([[2],[-1.5],[0.9]]) ,
                                           b = 0 ,
                                        )
y_grid_sinu_true = sinusoidal_fun(x_grid, frequency = frequency)

for lam in [0.0,10.0]:
    
    ## plot data and true model
    fig, (ax1,ax2) = plt.subplots(1,2, figsize = (10,5))

    ax1.plot(_X_train_poly, _t_train_poly , 'x', color = 'k', label = 'observations', zorder = 10)
    ax1.plot(x_grid, y_grid_poly_true, color = 'k', label = 'true function', zorder = 5)
    ax1.set_title(f'Polynomial data P = {poly_degree} \n $\lambda$ {lam}')
    ax1.legend()

    ax2.plot(_X_train_sinu, _t_train_sinu , 'x', color = 'k', label = 'observations', zorder = 10)
    ax2.plot(x_grid, y_grid_sinu_true, color = 'k', label = 'true function', zorder = 5)
    ax2.set_title(f'Sinusoidal data \n $\lambda$ {lam}')
    ax2.legend()
    
    for norm in [False,True]: 

        for i,poly_order in enumerate([4,20]):
            # grab original unormalized data
            X_train_poly, t_train_poly = copy.deepcopy(_X_train_poly), copy.deepcopy(_t_train_poly)
            X_train_sinu, t_train_sinu = copy.deepcopy(_X_train_sinu), copy.deepcopy(_t_train_sinu)

            
            ## generate polynomial features
            X_feat_poly = generate_features(X_train_poly, poly_order)
            X_feat_sinu = generate_features(X_train_sinu, poly_order)
            X_feat_grid_poly = generate_features(x_grid, poly_order)
            X_feat_grid_sinu = generate_features(x_grid, poly_order)
            
            if norm:
                # we normalize all but first column, which is a columns of ones.
                X_feat_poly, X_mean_poly, X_std_poly = norm_data(X_feat_poly[:,1:])
                X_feat_sinu, X_mean_sinu, X_std_sinu = norm_data(X_feat_sinu[:,1:])
                
                X_feat_grid_poly, _, _ = norm_data(X_feat_grid_poly[:,1:], mean = X_mean_poly, std = X_std_poly)
                X_feat_grid_sinu, _, _ = norm_data(X_feat_grid_sinu[:,1:], mean = X_mean_sinu, std = X_std_sinu)
                
                # concatenate back ones.
                X_feat_poly = np.hstack([np.ones((X_feat_poly.shape[0], 1)), X_feat_poly])
                X_feat_sinu = np.hstack([np.ones((X_feat_sinu.shape[0], 1)), X_feat_sinu])
                X_feat_grid_poly = np.hstack([np.ones((X_feat_grid_poly.shape[0], 1)), X_feat_grid_poly])
                X_feat_grid_sinu = np.hstack([np.ones((X_feat_grid_sinu.shape[0], 1)), X_feat_grid_sinu])
                
                t_train_poly, t_mean_poly, t_std_poly = norm_data(t_train_poly)
                t_train_sinu, t_mean_sinu, t_std_sinu = norm_data(t_train_sinu)
                
            # fit model to polynomial data
            w_opt_poly = fit_norm2_least_square( X_feat_poly , t_train_poly, lam = lam )

            # fit model to sinusoidal data
            w_opt_sinu = fit_norm2_least_square( X_feat_sinu, t_train_sinu, lam = lam )


            # draw function on fitted model
            y_grid_poly = computation_graph_linear(  
                                                   X_feat_grid_poly, 
                                                   w_opt_poly ,
                                                   b = 0 ,
                                                )

            y_grid_sinu = computation_graph_linear(  
                                               X_feat_grid_sinu, 
                                               w_opt_sinu ,
                                               b = 0 ,
                                            )
            # unormalize predictions
            if norm:
                y_grid_poly = y_grid_poly * t_std_poly + t_mean_poly
                y_grid_sinu = y_grid_sinu * t_std_sinu + t_mean_sinu
                
            linestyle = '-'
            if norm:
                linestyle = '--'

            ## plotting
            ax1.plot(x_grid, y_grid_poly , color = f'C{i+1}', linestyle = linestyle, label = f'order {poly_order} normalize {norm}')
            ax1.set_ylim([-1,16])
            ax1.legend()

            ax2.plot(x_grid, y_grid_sinu , color = f'C{i+1}', linestyle = linestyle, label = f'order {poly_order} normalize {norm}')
            ax2.set_ylim([-2,3])
            ax2.legend()


### $L_1$ regularization

$L_1$ regularization not only encourages the weights to be small, but also shrinks them to $0$. Thus, it is a way to perform automatic feature selection. For an intuitive description of why this happens, the following video is quite illustrative: https://www.youtube.com/watch?v=4qJrQ7DxAdk . Here is another interactive reference where you can have access to code to see why $L_1$ regularization has this behaviour: https://maitbayev.github.io/posts/why-l1-loss-encourage-coefficients-to-shrink-to-zero/. 

From the viewpoint of the Laplacian prior is quite understandable, because this prior distribution has a peak around $0$. From a geometrical viewpoint, the video is perhaps a better explanation of what can be done just with writing. From the optimization viewpoint, gradient-based optimization is very illustrative as well.

First of all, note that the $L_1$ norm is non-differentiable at $0$, which implies that at this point we need to use the subgradient. In practice, software such as PyTorch just uses the value $0$ for the subgradient at this point. This means that gradient descent on the $L_2$ and $L_1$ norms can be written as follows.

#### $L_2$

$$
\begin{split}
\grad{\wvec}{} L(\Xmat,\tvec,\wvec) = -2\cdot\Xmatt(\tvec - \Xmat\wvec)+ 2 \lambda \wvec
\end{split}
$$

which means gradient descent updates parameters as:

$$
\begin{split}
\wvec^{(t+1)} = \wvec^{(t)} - \alpha \bra{-\Xmatt(\tvec - \Xmat\wvec)+ \lambda \wvec}
\end{split}
$$

where the factor $2$ is absorbed by the learning rate. What do we observe here? Well, the rate of convergence of the parameter $\wvec$ to zero decreases when the parameter itself is close to zero, because $\lambda \wvec$ is small and so the modification made to the parameter when this parameter is close to $0$ is small. This means that when the parameter is close to zero, the gradient update is dominated by the data-dependent term.

#### $L_1$

With $L_1$ regularization, things are different. Note that the loss function is now:

$$
\begin{split}
L(\Xmat,\tvec,\wvec) = ||(\tvec-\Xmat\wvec)||^2_2; \text{  subject to  } ||\wvec||_1 \leq r
\end{split}
$$

In practice, the loss being minimized is given by:

$$
\begin{split}
L(\Xmat,\tvec,\wvec) = ||(\tvec-\Xmat\wvec)||^2_2 + \lambda ||\wvec||_1
\end{split}
$$

Now the square is not taken; otherwise the regularization term would always be positive. The reason why we take the square on the $L_2$ is for the gradient update to be constant (ie convexity with no change in curvature). Also, not taking the square of the norm implies a singularity at $w=0$. Finally, no exact solution to the minimization problem is available. **Exercise**

Since the gradient of the norm $1$ is constant, and (commonly) defined by $0$ at $w=0$, then gradient updates are given by:

$$
\begin{split}
\grad{\wvec}{} L(\Xmat,\tvec,\wvec) = -2\cdot\Xmatt(\tvec - \Xmat\wvec)+ \lambda \text{sign}(\wvec)
\end{split}
$$

which means gradient descent updates parameters as:

$$
\begin{split}
\wvec^{(t+1)} = \wvec^{(t)} - \alpha \bra{-2\cdot\Xmatt(\tvec - \Xmat\wvec)+ \lambda \text{sign}(\wvec)}
\end{split}
$$

we see that how much $\wvec$ is modified by the regularization term does not depend on $\wvec$. It will always be the value $1$ of $-1$, pushing the parameter towards zero, without depending on its magnitude. This means that the regularization term pushes equally to $0$ any parameter at any moment of the optimization.

However, state-of-the-art optimization of linear regression with $L_1$ regularization is done by a coordinate descent algorithm. This will allow us to introduce the skelarn library to fit this model.


#### Feature selection through $L_1$ regularization

Linear regression with $L_1$ regularization is also known as Lasso. Since the $L_1$ norm is non-differentiable at $x=0$, state-of-the-art optimization of this model is based on coordinate descent. Gradient descent could be used using the concept of subgradient. In fact, software such as PyTorch just defines the derivative at $x=0$ to be zero, which is one of the possible values for the subgradient. Nevertheless, this is slower than using coordinate descent.

$L_1$ regularization is able to select features from within the data; this means setting the coefficients to zero. For example, if:

$$
\begin{split}
y = b + w_1 \cdot x + w_2 \cdot x^2  + w_3 \cdot x^3
\end{split}
$$

but our data is linear, then $L_1$ regularization can set the coefficients $w_2,w_3$ to zero. This is also known as shrinkage. 

Any loss function can be regularized through $L_1$ regularization, since this is just a penalty we put on the parameters. For our polynomial regression model with squared loss, we have:

$$
\begin{split}
    L(t,x,w) = \sum^N_{n=1} \sum^K_{k=1} (t_n - (x_n^T)^k\cdot w_k)^2 + \sum^K_{k=1} \lambda_k || w_k ||_1
\end{split}
$$

where $\lambda$ denotes the strength of the regularizer, i.e how much we take it into account. We use the same $\lambda$ for any parameter.



##### The effect of normalization

Here, the effect of not normalizing the data is very visible. When the data is not normalized, since any coefficient is equally pushed to zero, the model tends to push low-order polynomial coefficients to zero more than high-order ones, even when the data-generating process is from a low-order polynomial.

To see this, assume we draw data from a second-order polynomial and fit a $ 5$th-order polynomial with $L_1$ regularization, with and without normalization, using the sklearn library.

Note that normalizing the targets changes the scale of the problem, and the $\lambda$ value needs to be decreased in this case as well. To illustrate the effect of normalization with $L_1$ regularization, I will not normalize the targets.

In [ ]:
%matplotlib inline
plt.close("all")

## Generate dataset
poly_degree_data = 2
noise_var = 0.0
N_data = 10
xmin = -3
xmax = 3

X_train_poly, t_train_poly = generate_polinomial_data(xmin, xmax, poly_degree_data, noise_var, N_data, seed = 1)

# ===========
## Train model

# model specification
poly_degree_model = 5  # assume our data follows a polynomial of order 10
X_train_feat = generate_features(X_train_poly, poly_degree=poly_degree_model, add_bias = False)

for reg_coeff in [0.1, 0.5]:

    print("================================")
    print("================================")
    print("================================")
    print(f"Regularization coefficient {reg_coeff}")

    ## Figure grid
    fig, ax_list  = plt.subplots(1,2,figsize = (10,5))
    
    for id_fig,to_norm_data in enumerate([True, False]):
        if to_norm_data:
            ## ================
            ## Normalized Model
        
            # Normalize data 
            scaler_X = StandardScaler()
            # scaler_T = StandardScaler()
            X_train_feat_scaled = scaler_X.fit_transform(X_train_feat)
            t_train_poly_scaled = t_train_poly #scaler_T.fit_transform(t_train_poly).ravel()
        else:
            X_train_feat_scaled = X_train_feat
            t_train_poly_scaled = t_train_poly 
            
        
        # model
        lasso = Lasso(alpha=reg_coeff)
        
        # optimize model
        lasso.fit(X_train_feat_scaled, t_train_poly_scaled)
        
        # Print coefficient values
        print("=============")
        if to_norm_data:
            print("Normalized")
        else:
            print("Un normalized")
            
        print("Coeficientes óptimos:", lasso.coef_)
        print("Intercepto:", lasso.intercept_)
        
        # ===========
        ## Plot regressed model
        N_grid = 100
        x_grid = np.linspace(-3,3,N_grid)
        X_grid_feat = generate_features(x_grid, poly_degree=poly_degree_model, add_bias = False)
        if to_norm_data:
            X_grid_feat_scaled = scaler_X.transform(X_grid_feat)  # escalar mismo rango
        else:
            X_grid_feat_scaled = X_grid_feat
        
        # Predecir con el model
        y_grid = lasso.predict(X_grid_feat_scaled)
        #y_grid = scaler_T.inverse_transform(y_grid.reshape(-1, 1)).ravel()
        
        # Graficar
        ax_list[id_fig].plot(X_train_poly, t_train_poly, 'o', color='C0', label='Train data')
        if to_norm_data:
            ax_list[id_fig].plot(x_grid, y_grid, color='C1', label='Normalized model')
        else:
            ax_list[id_fig].plot(x_grid, y_grid, color='C1', label='Unormalized model')
        ax_list[id_fig].set_xlabel('X')
        ax_list[id_fig].set_ylabel('t')
        ax_list[id_fig].set_title(f'Lasso Polynomial Fit (degree={poly_degree_model}, alpha={reg_coeff})')
        ax_list[id_fig].legend()
    


## Cross Validation

There are many questions that naturally arise from the current analysis of regularization.

* How do we optimally select the value for $\lambda$?
* How do we know if it is better to use $L_1$ or $L_2$ regularization?
* What degree of polynomial (or what model) should we use?
* How can we measure if the regularization used actually has an impact on the model performance?

My favourite answer to all these questions relies on Occam’s Razor and automatic model selection through marginal likelihoods, but this is often impractical for many applications, and a concept a bit difficult to understand when starting with Machine Learning. The idea basically relies on: "use the simpler model you can to explain your data". The marginal likelihood, automatically, encodes this information.

The answer we will see here is what is called cross-validation. The idea is to mimic a real scenario in which part of the data is used to train the model, and part of the data is used to validate the model. Note that a model is fitted to minimize a loss function over a set of data, and very expressive models can yield a nearly zero error on it. But how will it behave on new unseen data? The model we have fitted will need to explain this data well since it was not used to fit the model. We already observed how a very expressive model yields a very good training error but very poor test error.

Cross-validation exploits this idea by randomly partitioning the data into $M$ train-validation splits. The model is fitted on each train split and evaluated on its corresponding validation split. This is done for the $M$ models, and the one that has the lower error on average over the $M$ splits wins. 

Thus, we can test each parameter of our model using cross-validation to decide which is the configuration to use. Once we have decided, the model is fitted to the train data and is ready to measure on the test data. The test data mimics the future data your model will need to explain when it is working in a real environment. This pipeline can be easily done in sklearn for any model or parameter considered. To do so, consider the following functionalities: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html and https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html

### Manual cross-validation

Cross-validation can be easily implemented using loops and a performance metric. Suppose we use squared error as the score metric. Let's see how we can choose an optimal regularization parameter for the $L_2$ and $L_1$ models.

The left figure shows a polynomial of order $2$ fitted with some regularization $\lambda$ on the dataset. The right figure shows on each split. While running, we save the performance metric on the held-out set to later decide on the regularization term to be used.

In [ ]:
%matplotlib inline
plt.close("all")

## to draw model
fig, (ax1, ax2) = plt.subplots(1,2, figsize = (10,5))

## ====================
## for video generation

## temporary filename
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")


## Generate dataset
poly_degree_data = 2
noise_var = 0.5
N_data = 100
xmin = -3
xmax = 3

X_poly, t_poly = generate_polinomial_data(xmin, xmax, poly_degree_data, noise_var, N_data, seed = 1)

## grid for plotting
X_grid = np.linspace(-3,3,100)

## Model specs
# try 10 different lambda parameters
lambda_grid = np.linspace(0.1,10,5)
poly_degree_model = 10

X_poly_feats = generate_features(X_poly, poly_degree_model, add_bias = True)

# cross validation 
CV = 5

# try each paramteer CV times
loss = {}
valid_mean = []
train_mean = []
w_opts = []
for i,lam in enumerate(lambda_grid):
    
    loss[lam] = {
        'train' : [],
        'valid' : [],
    }

    loss_train = []
    loss_valid = []

    ## clear cross validation figure
    ax2.cla()
    
    for cv in range(CV):
        
        # Divide train-test
        X_temp, X_test, t_temp, t_test = train_test_split(
            X_poly, t_poly, test_size=0.2, random_state=cv
        )
        
        # Divide train validation
        X_train, X_valid, t_train, t_valid = train_test_split(
            X_temp, t_temp, test_size=0.25, random_state=cv
        )
    
    
        # Convert into polynomial features
        X_train_feats = generate_features(X_train, poly_degree_model, add_bias = True)
        X_valid_feats = generate_features(X_valid, poly_degree_model, add_bias = True)
        X_grid_feats = generate_features(X_grid, poly_degree_model, add_bias = True)

        # Fit model and save
        w_opt = fit_norm2_least_square(X_train_feats,t_train, lam = lam)

        # compute predictions on train, valid data and grid.
        y_train_feats = computation_graph_linear(  
                                               X_train_feats, 
                                               w_opt ,
                                               b = 0,
                                            )
        y_valid_feats = computation_graph_linear(  
                                               X_valid_feats, 
                                               w_opt ,
                                               b = 0,
                                            )

        y_grid = computation_graph_linear(  
                                               X_grid_feats, 
                                               w_opt ,
                                               b = 0,
                                            )

        # compute error on train and validation sets
        loss_train.append(float(np.sum(squared_loss_function(y_train_feats, t_train))))
        loss_valid.append(float(np.sum(squared_loss_function(y_valid_feats, t_valid))))

        # display function on each validation set
        if cv == 0:
            ax2.plot(X_poly, t_poly, '*', color = 'C0', label = 'all data')
            ax2.plot(X_train, t_train, '*', color = 'C1', label = 'train data')
            ax2.plot(X_valid, t_valid, '*', color = 'C2', label = 'valid data')
        else:
            ax2.plot(X_poly, t_poly, '*', color = 'C0')
            ax2.plot(X_train, t_train, '*', color = 'C1')
            ax2.plot(X_valid, t_valid, '*', color = 'C2')
            
        ax2.plot(x_grid, y_grid, color = f'C{cv+3}', label = f'$cv={cv}$')
        ax2.legend(loc = 'upper right')

        ## save video frame
        buf = BytesIO()
        fig.savefig(buf, format="png", dpi=100)
    
        buf.seek(0)
        frame = imageio.imread(buf) 
        writer.append_data(frame)  


    loss[lam]['train'] = loss_train
    loss[lam]['valid'] = loss_valid
    valid_mean.append(np.mean(loss_valid))
    train_mean.append(np.mean(loss_train))

    ## plot the trained model 
    w_opt = fit_norm2_least_square(X_poly_feats,t_poly, lam = lam)

    y_grid = computation_graph_linear(  
                                       X_grid_feats, 
                                       w_opt ,
                                       b = 0,
                                    )


    ax1.plot(X_poly, t_poly, '*', color = 'C0')
    ax1.plot(x_grid, y_grid, color = f'C{i+1}', label = f'$\lambda={lam}$')
    ax1.legend()

    ## save video frame
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)

    buf.seek(0)
    frame = imageio.imread(buf) 
    writer.append_data(frame)  
    
## Once computed, get the one with lowest validation.
idx_train = np.argmin(train_mean)
idx_val = np.argmin(valid_mean)

## Model selected
print(f"Model with lowest training error {lambda_grid[idx_train]:.2f} with train error {train_mean[idx_train]:.2f}")
print(f"Model selected lambda {lambda_grid[idx_val]:.2f} with validation error {valid_mean[idx_val]:.2f}")

writer.close() 
plt.close()

In [ ]:
# Mostrar el video en Jupyter Notebook
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

We now print train and validation errors for each split and polynomial order. What model would you choose?

In [ ]:
for k,v in loss.items():
    print("================")
    print(f"Regularization coefficient {k}")
    print(v)

Now, we perform the same experiment but in this case we vary the polynomial degree.

In [ ]:
%matplotlib inline
plt.close("all")

## to draw model
fig, (ax1, ax2) = plt.subplots(1,2, figsize = (10,5))
ax1.set_ylim([-2,20])

## ====================
## for video generation

## temporary filename
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")

## Generate dataset
poly_degree_data = 2
noise_var = 0.5
N_data = 30
xmin = -3
xmax = 3

X_poly, t_poly = generate_polinomial_data(xmin, xmax, poly_degree_data, noise_var, N_data, seed = 1)

## grid for plotting
X_grid = np.linspace(-3,3,100)

## Model specs
# try 10 different lambda parameters
poly_grid = [1,2,5,10,20]

# cross validation 
CV = 5

# try each paramteer CV times
loss = {}
valid_mean = []
train_mean = []
w_opts = []
for i,poly_degree_model in enumerate(poly_grid):
    
    X_poly_feats = generate_features(X_poly, poly_degree_model, add_bias = True)
    
    loss[poly_degree_model] = {
        'train' : [],
        'valid' : [],
    }

    loss_train = []
    loss_valid = []

    ## clear cross validation figure
    ax2.cla()
    ax2.set_ylim([-2,20])
    
    
    for cv in range(CV):
        
        # Divide train-test
        X_temp, X_test, t_temp, t_test = train_test_split(
            X_poly, t_poly, test_size=0.2, random_state=cv
        )
        
        # Divide train validation
        X_train, X_valid, t_train, t_valid = train_test_split(
            X_temp, t_temp, test_size=0.25, random_state=cv
        )
    
        # Convert into polynomial features
        X_train_feats = generate_features(X_train, poly_degree_model, add_bias = True)
        X_valid_feats = generate_features(X_valid, poly_degree_model, add_bias = True)
        X_grid_feats = generate_features(X_grid, poly_degree_model, add_bias = True)

        # Fit model and save
        w_opt = fit_norm2_least_square(X_train_feats,t_train, lam = 0.0)

        # compute predictions on train, valid data and grid.
        y_train_feats = computation_graph_linear(  
                                               X_train_feats, 
                                               w_opt ,
                                               b = 0,
                                            )
        y_valid_feats = computation_graph_linear(  
                                               X_valid_feats, 
                                               w_opt ,
                                               b = 0,
                                            )

        y_grid = computation_graph_linear(  
                                               X_grid_feats, 
                                               w_opt ,
                                               b = 0,
                                            )

        # compute error on train and validation sets
        loss_train.append(float(np.mean(squared_loss_function(y_train_feats, t_train))))
        loss_valid.append(float(np.mean(squared_loss_function(y_valid_feats, t_valid))))

        # display function on each validation set
        if cv == 0:
            ax2.plot(X_poly, t_poly, '*', color = 'C0', label = 'all data')
            ax2.plot(X_train, t_train, '*', color = 'C1', label = 'train data')
            ax2.plot(X_valid, t_valid, '*', color = 'C2', label = 'valid data')
        else:
            ax2.plot(X_poly, t_poly, '*', color = 'C0')
            ax2.plot(X_train, t_train, '*', color = 'C1')
            ax2.plot(X_valid, t_valid, '*', color = 'C2')
            
        ax2.plot(x_grid, y_grid, color = f'C{cv+3}', label = f'$cv={cv}$')
        ax2.legend(loc = 'upper right')

        
        ## save video frame
        buf = BytesIO()
        fig.savefig(buf, format="png", dpi=100)
    
        buf.seek(0)
        frame = imageio.imread(buf) 
        writer.append_data(frame)
        
    loss[poly_degree_model]['train'] = loss_train
    loss[poly_degree_model]['valid'] = loss_valid
    valid_mean.append(np.mean(loss_valid))
    train_mean.append(np.mean(loss_train))

    ## plot the trained model 
    w_opt = fit_norm2_least_square(X_poly_feats,t_poly, lam = 0.0)

    y_grid = computation_graph_linear(  
                                       X_grid_feats, 
                                       w_opt ,
                                       b = 0,
                                    )


    ax1.plot(X_poly, t_poly, '*', color = 'C0')
    ax1.plot(x_grid, y_grid, color = f'C{i+1}', label = f'$poly degree={poly_degree_model}$')
    ax1.legend()

    ## save video frame
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)

    buf.seek(0)
    frame = imageio.imread(buf) 
    writer.append_data(frame) 
    
## Once computed, get the one with lowest validation.
idx_train = np.argmin(train_mean)
idx_val = np.argmin(valid_mean)

## Model selected
print(f"Model with lowest training error poly degree = {poly_grid[idx_train]:.2f} with train error {train_mean[idx_train]:.2f}")
print(f"Model selected lambda poly degree = {poly_grid[idx_val]:.2f} with validation error {valid_mean[idx_val]:.2f}")

writer.close() 
plt.close()

In [ ]:
# Mostrar el video en Jupyter Notebook
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

We now print train and validation errors for each split and polynomial order. What model would you choose?

In [ ]:
for k,v in loss.items():
    print("================")
    print(f"Polynomial order {k}")
    print(v)

### Cross-validation with sklearn

Here is a code example on how to perform cross-validation with sklearn.

In [ ]:
## Generate dataset
poly_degree_data = 2
noise_var = 0.5
N_data = 10
xmin = -3
xmax = 3

X_train_poly, t_train_poly = generate_polinomial_data(xmin, xmax, poly_degree_data, noise_var, N_data, seed = 1)

# Pipeline of processing step till yield model fitting
pipeline = Pipeline([
    ("poly", PolynomialFeatures(include_bias=False)),
    ("scaler", StandardScaler()),
    ("reg", Ridge())  # Placeholder
])

# Hyperparameters
param_grid = [
    {   # L2
        "poly__degree": [1, 2, 3, 4, 5],
        "reg": [Ridge()],
        "reg__alpha": np.logspace(-3, 3, 7)
    },
    {   # L1
        "poly__degree": [1, 2, 3, 4, 5],
        "reg": [Lasso(max_iter=5000)],
        "reg__alpha": np.logspace(-3, 3, 7)
    }
]

# 10 fold cross validation
grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=10,
    scoring="neg_mean_absolute_percentage_error",
    n_jobs=-1
)

grid.fit(X_train_poly, t_train_poly)

print("Best parameters:", grid.best_params_)
print("Best scores:", grid.best_score_)
print("Best Model:", grid.best_estimator_)
print("Cross validation: ", pd.DataFrame(grid.cv_results_).keys())
print("Cross validation: ", pd.DataFrame(grid.cv_results_))

## Normalization

We have seen that normalization is relevant when regularization is going to be used. Beyond regularization, normalization is always relevant and a very important part of a Machine Learning pipeline.

While many LinkedIn "gurus" write many imprecise things about why normalization is important, and which types and when we use different types of normalizations, here we adopt a much simple an principle way to explain it.


Consider a problem $\mathbb{R}^2\to\mathbb{R}$ where no bias $b$ is present. Using the square loss function, we have:

$$
L(\Xmat,\tvec,\wvec) = \sum^N_{n=1}(t_n - w_1 x^n_1 - w_2 x^n_2)^2
$$

The gradient wrt each of the parameters is simply:

$$
\begin{split}
\grad{w_1}{} L(\Xmat,\tvec,\wvec) = -2\sum^N_{n=1}(t_n - w_1 x^n_1 - w_2 x^n_2)x^n_1\\
\grad{w_2}{} L(\Xmat,\tvec,\wvec) = -2\sum^N_{n=1}(t_n - w_1 x^n_1 - w_2 x^n_2)x^n_2
\end{split}
$$

Note that both gradients only differ in the term $(t_n - w_1 x^n_1 - w_2 x^n_2)$ being multiplied by either feature one $x^n_1$ or feature 2 $x^n_2$. Now, if we just think that the gradient is no more than the slope at a given point, then it is clear that if the relative scale of coordinate $1$ is much different from $2$, then the slope in coordinate $1$ will be much higher than coordinate $2$. For instance, if coordinate $x_1$ takes values around $100$ and coordinate $x_2$ around 1, then the slope over coordinate 1 will be $100$ times steeper than the other coordinate. This means that the loss function in one direction is much steeper than in the other. This has consequences regarding optimization.


Let's first visualize the effect of normalization in the loss function, and then we will talk about why this slope affects.

In [ ]:
np.random.seed(1)
N_data = 3
x_data = np.random.randn(N_data,2)* np.array([0.1,0.1]) + np.array([15,0]) 
t_data = np.zeros((N_data,1))

## To do so we need a mesh
N_points_domain = 100
w1, w2 = np.meshgrid(np.linspace(-5,5,N_points_domain),np.linspace(-5,5,N_points_domain))

# reshape for neural network
w_range = np.hstack((np.reshape(w1, (N_points_domain**2,1)),np.reshape(w2, (N_points_domain**2,1))))

# compute linear projection at all pairs of (w1,w2), for all data points at once
# x_data (N_data,2) @ w_range.T (2,P) -> (N_data,P)
y_pred = x_data @ w_range.T

# compute loss
expected_squared_loss_mesh = np.sum(squared_loss_function(t_data, y_pred), axis = 0)
expected_squared_loss_mesh = np.reshape(expected_squared_loss_mesh, (N_points_domain,N_points_domain))

fig = plt.figure(figsize=(10, 10))

ax1 = fig.add_subplot(2, 2, 1, projection="3d")
ax1.plot_surface(w1, w2, expected_squared_loss_mesh, cmap = "gray")
ax1.view_init(elev=40, azim=-90)

# normalized version of the inputs, same loss computation
x_data_norm, _, _ = norm_data(x_data)
y_pred_norm = x_data_norm @ w_range.T
expected_squared_loss_mesh_norm = np.sum(squared_loss_function(t_data, y_pred_norm), axis = 0)
expected_squared_loss_mesh_norm = np.reshape(expected_squared_loss_mesh_norm, (N_points_domain,N_points_domain))

ax2 = fig.add_subplot(2, 2, 2, projection="3d")
ax2.plot_surface(w1, w2, expected_squared_loss_mesh_norm, cmap = "gray")
ax2.view_init(elev=40, azim=-90)

ax3 = fig.add_subplot(2, 2, 3)
ax3.contourf(w1, w2, expected_squared_loss_mesh, cmap = "gray")

ax4 = fig.add_subplot(2, 2, 4)
ax4.contourf(w1, w2, expected_squared_loss_mesh_norm, cmap = "gray")


ax1.set_title("Unnormalized data")
ax2.set_title("Normalized data")

This difference in the shape of the loss function, being one direction much flatter than the other, has a direct impact on gradient-based optimization methods such as gradient descent. Second-order methods, or improved gradient descent methods such as momentum or Adam, which take gradient history to improve the updates, might improve upon the problem I am going to present. However, in models with more parameters, these problems tend to exacerbate, and these improvements are more effective if normalization is introduced or even unfeasible from a computational perspective.

The following video shows how gradient descent converges in both losses. We can see that optimizing over the loss that results from normalized data is more stable and enables similar learning rates on both coordinates. Since the loss that results from unnormalized data has very different slopes, we need to adjust the learning rate in both directions for successful convergence.

In [ ]:
w_orig = np.array([[3.0],[3.0]])
w_norm = np.array([[3.0],[3.0]])

lr_unnorm = 0.00140
lr_norm = 0.1

epochs = 20

## keep the full history of visited points, to show the whole trajectory
## (and its oscillations) instead of just the last step
y_orig0 = computation_graph_linear(x_data, w_orig, 0)
loss_orig0 = np.sum(squared_loss_function(t_data, y_orig0))
y_norm0 = computation_graph_linear(x_data_norm, w_norm, 0)
loss_norm0 = np.sum(squared_loss_function(t_data, y_norm0))

history_w_orig = [np.squeeze(w_orig)]
history_loss_orig = [loss_orig0]
history_w_norm = [np.squeeze(w_norm)]
history_loss_norm = [loss_norm0]

video_filename = "/tmp/aux.mp4"
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")

fig = plt.figure(figsize=(10, 10))
ax1 = fig.add_subplot(2, 2, 1, projection="3d")
ax2 = fig.add_subplot(2, 2, 2, projection="3d")
ax3 = fig.add_subplot(2, 2, 3)
ax4 = fig.add_subplot(2, 2, 4)

for e in range(epochs):

    ## forward plus backward, unnormalized
    grad_w_orig, _ = grad_squared_loss_wrt_linear_model(x_data, t_data, w_orig, 0)
    w_orig_n = w_orig - lr_unnorm * grad_w_orig
    y_orig_n = computation_graph_linear(x_data, w_orig_n, 0)
    loss_orig_n = np.sum(squared_loss_function(t_data, y_orig_n))

    ## forward plus backward, normalized
    grad_w_norm, _ = grad_squared_loss_wrt_linear_model(x_data_norm, t_data, w_norm, 0)
    w_norm_n = w_norm - lr_norm * grad_w_norm
    y_norm_n = computation_graph_linear(x_data_norm, w_norm_n, 0)
    loss_norm_n = np.sum(squared_loss_function(t_data, y_norm_n))

    history_w_orig.append(np.squeeze(w_orig_n))
    history_loss_orig.append(loss_orig_n)
    history_w_norm.append(np.squeeze(w_norm_n))
    history_loss_norm.append(loss_norm_n)

    ## ============= ##
    ## START DRAWING ##
    ## ============= ##
    ax1.clear(); ax2.clear(); ax3.clear(); ax4.clear()

    n_orig = len(history_w_orig)
    n_norm = len(history_w_norm)

    ## --- unnormalized, 3d: full trajectory, lines except the last update (arrow) ---
    ax1.plot_surface(w1, w2, expected_squared_loss_mesh, cmap="gray")
    for i in range(n_orig):
        wi = history_w_orig[i]
        ax1.plot(wi[0], wi[1], history_loss_orig[i], 'o', color='C1', markersize=6, zorder=50)
        if i > 0:
            wp = history_w_orig[i-1]
            if i == n_orig - 1:
                arrow = Arrow3D([wp[0], wi[0]], [wp[1], wi[1]], [history_loss_orig[i-1], history_loss_orig[i]],
                                 mutation_scale=15, lw=1, arrowstyle="-|>", color="C1")
                ax1.add_artist(arrow)
            else:
                ax1.plot([wp[0], wi[0]], [wp[1], wi[1]], [history_loss_orig[i-1], history_loss_orig[i]], color='C1')
    ax1.view_init(elev=40, azim=-90)
    ax1.set_title("Unnormalized features")

    ## --- normalized, 3d: full trajectory, lines except the last update (arrow) ---
    ax2.plot_surface(w1, w2, expected_squared_loss_mesh_norm, cmap="gray")
    for i in range(n_norm):
        wi = history_w_norm[i]
        ax2.plot(wi[0], wi[1], history_loss_norm[i], 'o', color='C1', markersize=6, zorder=50)
        if i > 0:
            wp = history_w_norm[i-1]
            if i == n_norm - 1:
                arrow = Arrow3D([wp[0], wi[0]], [wp[1], wi[1]], [history_loss_norm[i-1], history_loss_norm[i]],
                                 mutation_scale=15, lw=1, arrowstyle="-|>", color="C1")
                ax2.add_artist(arrow)
            else:
                ax2.plot([wp[0], wi[0]], [wp[1], wi[1]], [history_loss_norm[i-1], history_loss_norm[i]], color='C1')
    ax2.view_init(elev=40, azim=-90)
    ax2.set_title("Normalized features")

    ## --- unnormalized, contour: full trajectory, lines except the last update (arrow) ---
    ax3.contourf(w1, w2, expected_squared_loss_mesh, cmap="gray")
    for i in range(n_orig):
        wi = history_w_orig[i]
        ax3.plot(wi[0], wi[1], 'o', color='C1')
        if i > 0:
            wp = history_w_orig[i-1]
            if i == n_orig - 1:
                ax3.arrow(wp[0], wp[1], wi[0] - wp[0], wi[1] - wp[1], color="C1", width=0.02,
                          head_width=0.15, length_includes_head=True)
            else:
                ax3.plot([wp[0], wi[0]], [wp[1], wi[1]], color='C1')

    ## --- normalized, contour: full trajectory, lines except the last update (arrow) ---
    ax4.contourf(w1, w2, expected_squared_loss_mesh_norm, cmap="gray")
    for i in range(n_norm):
        wi = history_w_norm[i]
        ax4.plot(wi[0], wi[1], 'o', color='C1')
        if i > 0:
            wp = history_w_norm[i-1]
            if i == n_norm - 1:
                ax4.arrow(wp[0], wp[1], wi[0] - wp[0], wi[1] - wp[1], color="C1", width=0.02,
                          head_width=0.15, length_includes_head=True)
            else:
                ax4.plot([wp[0], wi[0]], [wp[1], wi[1]], color='C1')

    fig.suptitle(f"epoch {e+1}/{epochs}")

    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    buf.seek(0)
    frame = imageio.imread(buf)
    writer.append_data(frame)

    w_orig = w_orig_n
    w_norm = w_norm_n

writer.close()
plt.close()

In [ ]:
# Mostrar el video en Jupyter Notebook
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

### Types of normalization

* Z-score: removing the mean and dividing by the standard deviation. It centers the distribution around zero. Having data around zero is relevant for gradient descent (I can't remember why, but it is written in a 1998 Yann LeCun paper)
* Min-max: Data is normalized so that the minimum is at zero and the maximum at 1, or any other interval. Typycal ones are $[0,1]$ and [-1,1].
* Distribution transformations: these are transformations that, when applied, change the distribution of the data. One important one is the box-cox or the logarithm, which aim at making the data more Gaussian.


### Normalization to avoid misspecification.

We have seen that normalization is relevant for two purposes. First, it allows regularization to do its job properly. Second, it reshapes the loss function so that gradient-guided methods are more robust. This has a direct consequence in the number of iterations needed for the model to converge properly, and also makes the initialization of the model parameters and the learning rate more robust.

 There is another thing normalization is useful for: removing misspecification. For instance, think in the distribution of house pricing. This distribution is very skewed, with a long right tail. It is also asymmetric. Applying the log to this distribution makes it more Gaussian. Why is it important? Well, as we will see in the next chapter, the linear models, with the squared loss, assume the data is Gaussian with constant variance, where the mean is actually predicted by the linear model.

 Also, even if we make it Gaussian, the range of the the output might affect convergence.  Thus, we can combine the log plus z-score normalization. This ensures we have an approximately standard Gaussian distribution and thus gradients behave well.

In [ ]:
rng = np.random.default_rng(42)
n = 300
size_m2 = rng.uniform(40, 250, size=n)

beta0, beta1, sigma = 10.5, 0.006, 0.15
log_price = beta0 + beta1 * size_m2 + rng.normal(0, sigma, size=n)
price = np.exp(log_price)  # before normalizing
price_normalized = log_price  # after normalizing (log-gaussianized)
price_zscore = (price_normalized - price_normalized.mean()) / price_normalized.std()  # z-score on top

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

axes[0].hist(price, bins=25, density=True, color="C0", edgecolor="black", alpha=0.7)
axes[0].set_xlabel("price")
axes[0].set_ylabel("density")
axes[0].set_title("Before normalizing: right-skewed")

axes[1].hist(price_normalized, bins=25, density=True, color="C1", edgecolor="black", alpha=0.7)
grid = np.linspace(price_normalized.min(), price_normalized.max(), 200)
axes[1].plot(grid, scipy_norm.pdf(grid, price_normalized.mean(), price_normalized.std()), color="black", linewidth=2.5, label="Gaussian fit")
axes[1].set_xlabel("log(price)")
axes[1].set_ylabel("density")
axes[1].set_title("After log: gaussianized")
axes[1].legend()

axes[2].hist(price_zscore, bins=25, density=True, color="C2", edgecolor="black", alpha=0.7)
grid_z = np.linspace(price_zscore.min(), price_zscore.max(), 200)
axes[2].plot(grid_z, scipy_norm.pdf(grid_z, 0, 1), color="black", linewidth=2.5, label=r"$\mathcal{N}(0,1)$")
axes[2].set_xlabel("z-score(log(price))")
axes[2].set_ylabel("density")
axes[2].set_title("After z-score: standardized")
axes[2].legend()

for ax in axes:
    ax.grid(True)

plt.tight_layout()
plt.show()

Gaussianizing the *marginal* distribution of $y$, as we just did above, is not really the point. The linear model with squared loss (as we will see in the next chapter) assumes a *conditional* distribution, $p(y\mid x) = \mathcal{N}(\mu(x),\sigma^2)$ with $\mu(x)$ linear in $x$ and $\sigma^2$ constant across $x$. Thus, it is $p(y\mid x)$, not the marginal $p(y)$, that must be well specified for those assumptions to hold. A Gaussian-looking histogram of $y$ (or $\log y$) on its own says nothing about whether this conditional structure actually holds, since the marginal integrates $x$ out entirely. So let's check the thing that actually matters directly: plot $y$ against $x$ before and after normalizing, and see whether the relationship looks like what the model assumes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

axes[0].scatter(size_m2, price, color="C0", edgecolor="black", alpha=0.7)
axes[0].set_xlabel("size (m2)")
axes[0].set_ylabel("price")
axes[0].set_title("Before normalizing: price vs size")

axes[1].scatter(size_m2, price_normalized, color="C1", edgecolor="black", alpha=0.7)
axes[1].set_xlabel("size (m2)")
axes[1].set_ylabel("log(price)")
axes[1].set_title("After log: log(price) vs size")

axes[2].scatter(size_m2, price_zscore, color="C2", edgecolor="black", alpha=0.7)
axes[2].set_xlabel("size (m2)")
axes[2].set_ylabel("z-score(log(price))")
axes[2].set_title("After z-score: z-score(log(price)) vs size")

for ax in axes:
    ax.grid(True)

plt.tight_layout()
plt.show()


#### And why is normalization helping here?

Look at the noise model behind `price`: it was generated as `log_price = beta0 + beta1*size_m2 + noise` and then `price = exp(log_price)`, i.e. $\text{price} = \exp(\beta_0+\beta_1\cdot\text{size})\cdot\exp(\epsilon)$ with $\epsilon\sim\mathcal{N}(0,\sigma^2)$ being multiplicative noise. In the raw scale this is heteroscedastic: the absolute spread of `price` around its mean grows with the mean itself (a $1$M house has a much larger absolute-dollar noise budget than a $100$k one, even though the relative noise, $\exp(\epsilon)$, is the same for both). Taking $\log$ turns this multiplicative noise into additive noise, $\log(\text{price}) = \beta_0+\beta_1\cdot\text{size}+\epsilon$, which is homoscedastic by construction: the same $\sigma^2$ regardless of `size`. That is exactly what the middle panel above shows: `log(price)` vs. `size` scatters with a constant width around a straight line, unlike the fan-shaped, ever-widening scatter of raw `price` vs. `size` in the left panel.

This does not mean that heteroscedasticity can always be removed by this proceedure.

## TODO

* Move normalization to a separate section. Here, explain what happens under multicollinearity as well. Both on gradient descent methods and OLS. Also include why not normalizing the data affects the OLS solution, making it less stable. Yann LeCun had a 90s paper where he talks about what happens in this scenario of unnormalized and dependent data in the loss function. I think he mentioned something like there are zones of constant loss where the algorithm might get stuck.